---

# Forecasting + Text End-to-End Notebook (EDA + Pipeline)

*Time-series and text exploratory analysis, the text lead analysis, and the forecasting pipeline with text features in one run. Part 1 explores the data and writes `eda-output/eda_decisions.json`; Part 2 reads it, so every `'auto'` setting in the pipeline follows the EDA findings. Outputs go to `eda-output/` and `pipeline-output/`.*

---

## Team Name

- Name 1 (Role)
- Name 2
- Name 3

---

## Table of Contents

**Part 1: Exploratory Data Analysis**

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Toolkit**](#3)
4. [**Structure and Frequency**](#4)
5. [**Completeness and Intermittency**](#5)
6. [**Target Distribution and Transform**](#6)
7. [**Series Overview**](#7)
8. [**Decomposition and Strength**](#8)
9. [**Seasonal Patterns**](#9)
10. [**Autocorrelation and Lags**](#10)
11. [**Stationarity**](#11)
12. [**Anomalies and Level Shifts**](#12)
13. [**Exogenous and Calendar Drivers**](#13)
14. [**Cross-Series Structure**](#14)
15. [**Baseline Forecastability**](#15)
16. [**Text Signal**](#16)
17. [**EDA Summary and Pipeline Decisions**](#17)

**Part 2: Modelling Pipeline** (reads `eda-output/eda_decisions.json` written by Part 1)

18. [**Pipeline Initialization**](#18)
19. [**Toolkit**](#19)
20. [**EDA-Driven Decisions**](#20)
21. [**Data Preparation**](#21)
22. [**Feature Engineering**](#22)
23. [**Backtesting Design**](#23)
24. [**Statistical Baselines**](#24)
25. [**Gradient Boosting Forecasters**](#25)
26. [**Ensemble and Model Selection**](#26)
27. [**Backtest Diagnostics**](#27)
28. [**Model Interpretation**](#28)
29. [**Final Forecast**](#29)
30. [**Export**](#30)

---

# Introduction <a name="1"></a>

---

## Overview

*Forecasting errors usually come from misunderstanding the data, not from the model: a wrong frequency, unhandled gaps, a missed seasonality, intermittent demand treated as smooth, or a validation split that leaks the future. This notebook answers those questions systematically before any model is trained.*

## Aim

*The notebook identifies the frequency, the panel structure, gaps and intermittency, the right variance-stabilising transform, trend and seasonal strength, dominant periods, autocorrelation lags, stationarity, anomalies and level shifts, exogenous and calendar effects, cross-series similarity, and how hard the series are to beat with naive forecasts. It ends with concrete pipeline decisions: horizon, seasonal periods, lag and rolling features, transform, backtesting scheme, metric, and model family.*

## How to Point the Notebook at a Dataset

*All changes happen in `Settings` (section 2). Data must be in long format: one row per series and timestamp.*

| Setting | Purpose | Example |
|---|---|---|
| `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` | Training file per environment | `'/kaggle/input/comp'` (folder) or a file path |
| `TEST_*_PATH` | Optional future frame; its dates define the forecast horizon | `'.../test.csv'` |
| `DATE_COL`, `TARGET_COL` | Timestamp and value to forecast | `'date'`, `'sales'` |
| `ID_COLS` | Columns identifying a series; `[]` for a single series | `['store', 'item']` |
| `FREQ` | Pandas frequency, or `'auto'` to infer | `'D'`, `'W-MON'`, `'MS'`, `'h'` |
| `EXOG_COLS` | Known covariates, or `'auto'` for all other numeric columns | `['price', 'promo']` |
| `HORIZON` | Steps to forecast, or `'auto'` (from the test file, else a frequency default) | `28` |

*When the configured file does not exist and `DEMO_IF_MISSING = True`, a realistic synthetic retail dataset is generated so the notebook can be explored end to end.*

## Dataset

*Any time series in long format works: one row per date (and per series), a date column, a numeric target, optional series identifiers (store, item, region), and optional exogenous columns (price, promotion, weather). Put the files in `data/` next to this notebook, or point `KAGGLE_PATH` / `COLAB_PATH` / `LOCAL_PATH` at the dataset folder.*

```
data/
├── train.csv              (date, ids, target, exogenous)
├── test.csv               (future dates to forecast, optional)
└── sample_submission.csv  (id + target columns, optional)
```

*With `'auto'` the date column is recognised by name, the target comes from `sample_submission.csv` (or is the only train column missing from the test file), and the remaining text columns become series identifiers. When no data is found and `DEMO_IF_MISSING = True`, a synthetic retail panel is generated in `demo-data/` so the notebook still runs end to end.*

## Approach: Diagnostic Ladder

*Each step answers one modelling question, and its answer feeds the decisions file.*

```
Long table (id, date, target, exog)
    |
    +-- Structure      -> frequency, number of series, lifespans          -> FREQ, HORIZON
    +-- Completeness   -> gaps, zeros, ADI / CV^2 intermittency           -> fill strategy, Croston/TSB
    +-- Distribution   -> skew, variance-level slope                      -> log / none transform
    +-- Decomposition  -> STL trend and seasonal strength                 -> seasonal models
    +-- Seasonality    -> profiles, periodogram                           -> seasonal periods
    +-- Autocorrelation-> ACF / PACF significant lags                     -> lag features
    +-- Stationarity   -> ADF + KPSS, differencing order                  -> ARIMA d, detrending
    +-- Anomalies      -> STL residual outliers, level shifts             -> cleaning, training window
    +-- Drivers        -> lagged cross-correlation, calendar effects      -> exogenous features
    +-- Cross-series   -> correlation, Pareto of volume                   -> global vs local models
    +-- Baselines      -> naive / seasonal naive backtest (MASE, WAPE)    -> the score to beat
         |
    eda_summary.md + eda_decisions.json
```

---

# Initialization <a name="2"></a>

---

## Environment Setup

The notebook runs on CPU only. The `nvidia-smi` call is kept for consistency with the other templates and fails harmlessly on CPU machines.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scipy scikit-learn statsmodels pyarrow openpyxl

## Import Libraries

In [ ]:
import os
import re
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import signal, stats
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

All paths, column roles, frequency, horizon, and analysis knobs are centralised here. Switching between Kaggle, Colab, and local execution only requires editing the path lines.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}
    DEMO_IF_MISSING  = True

    # 2) Columns
    DATE_COL   = 'auto'
    TARGET_COL = 'auto'
    ID_COLS    = 'auto'
    EXOG_COLS  = 'auto'
    AGG        = 'sum'
    # 2b) Text modality: a text column of the main table, or a separate dated text file (news, reviews, posts)
    TEXT_KAGGLE_PATH = None
    TEXT_COLAB_PATH  = None
    TEXT_LOCAL_PATH  = None
    TEXT_COL         = 'auto'
    TEXT_DATE_COL    = 'auto'
    TEXT_ID_COLS     = 'auto'
    TEXT_DIM         = 8
    TEXT_LAG         = 'auto'

    # 3) Time settings
    FREQ             = 'auto'
    HORIZON          = 'auto'
    SEASONAL_PERIODS = 'auto'

    # 4) Analysis knobs
    MAX_SERIES_PLOT = 12
    MAX_SERIES_STAT = 500
    MAX_LAGS        = 'auto'
    OUTLIER_Z       = 4.0

    # 5) Outputs
    DATA_PATH  = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH  = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    TEXT_PATH  = TEXT_KAGGLE_PATH if _ON_KAGGLE else TEXT_COLAB_PATH if _ON_COLAB else TEXT_LOCAL_PATH
    OUTPUT_DIR = Path('/kaggle/working/eda-output') if _ON_KAGGLE else Path('eda-output')
    FIG_DIR    = OUTPUT_DIR / 'figures'
    TABLE_DIR  = OUTPUT_DIR / 'tables'

CFG = Settings()
for d in (CFG.OUTPUT_DIR, CFG.FIG_DIR, CFG.TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Test path   : {CFG.TEST_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')

## Demo Data Generator

The generator builds a long-format retail panel that contains every pattern the notebook is designed to detect. It only runs when the configured training file is missing and `DEMO_IF_MISSING` is true, so it never overwrites real data.

In [ ]:
def make_demo(train_path, test_path, seed=42):
    rng = np.random.RandomState(seed)
    dates = pd.date_range('2021-01-01', '2024-06-30', freq='D')
    future = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=28, freq='D')
    lebaran = pd.to_datetime(['2021-05-13', '2022-05-02', '2023-04-22', '2024-04-10'])
    all_dates = dates.append(future)
    t = np.arange(len(all_dates))
    weekly = np.array([0.9, 0.85, 0.9, 0.95, 1.1, 1.35, 1.25])[all_dates.dayofweek]
    yearly = 1 + 0.15 * np.sin(2 * np.pi * (all_dates.dayofyear - 80) / 365.25)
    days_to_eid = np.min(np.abs((all_dates.values[:, None] - lebaran.values[None, :]).astype('timedelta64[D]').astype(int)), axis=1)
    eid = 1 + 1.2 * np.exp(-days_to_eid / 4.0) * (days_to_eid <= 14)
    rows, news = [], []
    words = 'cuaca hari ini kota ramai jalan macet pasar harga stabil warga libur sekolah acara musik'.split()
    for s in range(1, 7):
        # store announcements lift sales 30 to 36 days later, longer than the 28-day horizon
        ann = np.where((rng.rand(len(t)) < 0.03) & (t < len(dates)))[0]
        boost = np.ones(len(t))
        for i in ann:
            boost[i + 30:i + 37] *= 1.7
            news.append({'date': all_dates[i], 'store': f'S{s}', 'headline': f'toko S{s} umumkan diskon besar dan promo spesial bulan depan'})
        for i in np.where(rng.rand(len(dates)) < 0.6)[0]:
            news.append({'date': dates[i], 'store': f'S{s}', 'headline': ' '.join(rng.choice(words, 7))})
        for it in range(1, 6):
            base = rng.uniform(20, 120) if it <= 3 else rng.uniform(0.3, 1.5)
            trend = 1 + rng.uniform(-0.1, 0.35) * t / len(t)
            price0 = rng.uniform(10, 60)
            price = price0 * (1 + 0.04 * np.sin(t / 90 + s)) * np.where(rng.rand(len(t)) < 0.03, 0.85, 1.0)
            promo = (rng.rand(len(t)) < 0.06).astype(int)
            lam = base * trend * weekly * yearly * eid * (1 + 0.5 * promo) * (price / price0) ** -1.5 * boost
            if s == 3 and it == 1:
                lam = lam * np.where(all_dates >= '2023-03-01', 1.6, 1.0)
            y = rng.poisson(lam * rng.gamma(20, 1 / 20, len(t))).astype(float)
            frame = pd.DataFrame({'date': all_dates, 'store': f'S{s}', 'item': f'I{it}', 'sales': y,
                                  'price': price.round(2), 'promo': promo})
            if s == 6:
                frame = frame[frame['date'] >= '2022-07-01']
            rows.append(frame)
    full = pd.concat(rows, ignore_index=True)
    train = full[full['date'] <= dates[-1]].copy()
    spikes = train.sample(12, random_state=seed).index
    train.loc[spikes, 'sales'] *= 8
    train = train.drop(train[(train['date'].between('2022-02-10', '2022-02-16')) & (train['store'] == 'S2')].index)
    train = train.drop(train.sample(frac=0.01, random_state=seed).index)
    test = full[full['date'] > dates[-1]].drop(columns='sales')
    Path(train_path).parent.mkdir(parents=True, exist_ok=True)
    train.to_csv(train_path, index=False)
    test.to_csv(test_path, index=False)
    pd.DataFrame(news).to_csv(Path(train_path).parent / 'news.csv', index=False)
    print(f'Demo data written: {train_path} ({len(train):,} rows), {test_path} ({len(test):,} rows)')

## Load Dataset

The file is resolved (with the Kaggle fallback to the largest tabular file), loaded, and the date column parsed. Duplicate rows for the same series and timestamp are aggregated with `AGG`, and every series receives a single string key.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = [f for f in r.rglob('*') if f.is_file() and f.suffix.lower() in TABULAR_EXT
                     and not SKIP_DIRS & set(f.parts)]
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, CFG.TEST_PATH)
if DATA_PATH is None and CFG.DEMO_IF_MISSING:
    make_demo('demo-data/train.csv', 'demo-data/test.csv', CFG.SEED)
    DATA_PATH = Path('demo-data/train.csv')
    TEST_PATH = Path('demo-data/test.csv')
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found. Edit the path lines in Settings.'
if TEST_PATH is None:
    print('[warn] No test file configured or found in the dataset folder: the horizon falls back to the frequency default.')

raw = load_table(DATA_PATH, **CFG.READ_KWARGS)
raw_test = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
def resolve_columns():
    """Fill 'auto' columns: date by name, target from sample_submission or the train-only column, ids = text columns."""
    if CFG.DATE_COL == 'auto':
        named = [c for c in raw.columns if 'date' in c.lower() or 'time' in c.lower() or c.lower() in ('ds', 'period', 'month', 'week', 'day', 'tanggal')]
        CFG.DATE_COL = named[0] if named else raw.columns[0]
    if CFG.TARGET_COL == 'auto':
        sample = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
        only_train = [c for c in raw.columns if raw_test is not None and c not in raw_test.columns]
        if sample is not None and sample.shape[1] >= 2 and sample.columns[1] in raw.columns:
            CFG.TARGET_COL = sample.columns[1]
        elif len(only_train) == 1:
            CFG.TARGET_COL = only_train[0]
        else:
            CFG.TARGET_COL = [c for c in raw.columns if c != CFG.DATE_COL and pd.api.types.is_numeric_dtype(raw[c])][-1]
    if CFG.ID_COLS == 'auto':
        CFG.ID_COLS = [c for c in raw.columns if c not in (CFG.DATE_COL, CFG.TARGET_COL) and not pd.api.types.is_numeric_dtype(raw[c]) and raw[c].dropna().astype(str).head(2000).str.split().str.len().mean() < 3]
    print(f'Columns: date {CFG.DATE_COL!r}, target {CFG.TARGET_COL!r}, series ids {CFG.ID_COLS}')

resolve_columns()
D, Y = CFG.DATE_COL, CFG.TARGET_COL
IDS = [c for c in CFG.ID_COLS if c in raw.columns]

def prepare(frame):
    frame = frame.copy()
    frame[D] = pd.to_datetime(frame[D], errors='coerce')
    frame = frame.dropna(subset=[D])
    frame['series'] = frame[IDS].astype(str).agg(' | '.join, axis=1) if IDS else 'total'
    return frame

raw = prepare(raw)
raw_test = prepare(raw_test) if raw_test is not None else None
EXOG = [c for c in raw.columns if c not in IDS + [D, Y, 'series'] and pd.api.types.is_numeric_dtype(raw[c])] \
    if CFG.EXOG_COLS == 'auto' else [c for c in CFG.EXOG_COLS if c in raw.columns]

dups = int(raw.duplicated(['series', D]).sum())
agg_map = {Y: CFG.AGG, **{c: 'mean' for c in EXOG}}
df = raw.groupby(['series', D], as_index=False).agg(agg_map) if dups else raw[['series', D, Y] + EXOG].copy()
df = df.sort_values(['series', D]).reset_index(drop=True)
print(f'Loaded {len(raw):,} rows from {DATA_PATH}; {dups:,} duplicate (series, date) rows aggregated with {CFG.AGG}')
print(f'Series: {df.series.nunique():,} | id columns: {IDS} | exogenous: {EXOG}')
if raw_test is not None:
    print(f'Test frame: {len(raw_test):,} rows, {raw_test[D].min().date()} to {raw_test[D].max().date()}')
display(df.head())

---

# Toolkit <a name="3"></a>

---

The plotting identity matches the other templates. Date axes use concise automatic formatting, and `save_fig` writes every figure into `eda-output/figures/` with a running number.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])
CMAP_DIV  = LinearSegmentedColormap.from_list('div', ['#EE6C4D', '#FBE3DC', '#F7F7F7', '#D6E4F0', '#3D5A80'])
CMAP_HEAT = LinearSegmentedColormap.from_list('heat', ['#FFF8F0', '#F4A261', '#E76F51', '#9D0208'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = [0]
SAVED_FIGS = []

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name):
    FIG_COUNTER[0] += 1
    path = CFG.FIG_DIR / f'{FIG_COUNTER[0]:02d}_{slug(name)}.png'
    fig.savefig(path, facecolor='white')
    SAVED_FIGS.append(path.name)
    plt.show()
    plt.close(fig)

def save_table(table, name, index=True):
    table.to_csv(CFG.TABLE_DIR / f'{slug(name)}.csv', index=index)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def date_axis(ax):
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))

def bar_labels(ax, fmt='{:,.0f}', size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues], padding=3, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

---

# Structure and Frequency <a name="4"></a>

---

The frequency is the most fundamental decision in forecasting. It is inferred from the most common gap between consecutive timestamps within each series, because a few missing dates make `pd.infer_freq` fail. The frequency then sets the default seasonal periods and horizon. The series lifespan chart shows when each series starts and ends, which reveals new and discontinued series (cold-start cases).

In [ ]:
OFFSET_DEFAULTS = {'min': ([60, 1440], 60), 'h': ([24, 168], 48), 'D': ([7, 365], 28), 'W': ([52], 13),
                   'MS': ([12], 12), 'M': ([12], 12), 'QS': ([4], 8), 'Q': ([4], 8), 'YS': ([1], 3), 'Y': ([1], 3)}

def infer_freq(frame):
    diffs = frame.groupby('series')[D].diff().dropna()
    step = diffs.mode().iloc[0]
    days = step / pd.Timedelta(days=1)
    if days < 1 / 24:
        return f'{int(step / pd.Timedelta(minutes=1))}min', 'min'
    if days < 1:
        hours = int(round(days * 24))
        return ('h' if hours == 1 else f'{hours}h'), 'h'
    if days < 6:
        return 'D', 'D'
    if days < 25:
        dow = frame[D].dt.dayofweek.mode().iloc[0]
        return f'W-{["MON", "TUE", "WED", "THU", "FRI", "SAT", "SUN"][dow]}', 'W'
    if days < 80:
        return ('MS' if (frame[D].dt.day == 1).mean() > 0.8 else 'ME'), 'MS'
    if days < 300:
        return 'QS', 'QS'
    return 'YS', 'YS'

if CFG.FREQ == 'auto':
    FREQ, FREQ_KEY = infer_freq(df)
else:
    FREQ = CFG.FREQ
    FREQ_KEY = next((k for k in OFFSET_DEFAULTS if FREQ.upper().startswith(k.upper())), 'D')
default_periods, default_h = OFFSET_DEFAULTS.get(FREQ_KEY, ([7], 14))

life = df.groupby('series').agg(start=(D, 'min'), end=(D, 'max'), n_obs=(D, 'size'), total=(Y, 'sum'))
full_index = pd.date_range(df[D].min(), df[D].max(), freq=FREQ)
life['expected'] = [len(pd.date_range(s, e, freq=FREQ)) for s, e in zip(life['start'], life['end'])]
life = life.sort_values('total', ascending=False)

if CFG.HORIZON != 'auto':
    HORIZON, H_SOURCE = int(CFG.HORIZON), 'user'
elif raw_test is not None:
    HORIZON, H_SOURCE = int(raw_test.groupby('series')[D].nunique().median()), 'test file'
else:
    HORIZON, H_SOURCE = default_h, 'frequency default'

structure = pd.Series({
    'frequency': FREQ, 'start': df[D].min().date(), 'end': df[D].max().date(), 'timestamps': len(full_index),
    'series': df['series'].nunique(), 'observations': len(df), 'median series length': int(life['n_obs'].median()),
    'shortest series': int(life['n_obs'].min()), 'series starting late': int((life['start'] > df[D].min()).sum()),
    'series ending early': int((life['end'] < df[D].max()).sum()), 'horizon': f'{HORIZON} ({H_SOURCE})',
}, name='value').to_frame()
save_table(structure, 'structure')
save_table(life, 'series_lifespans')
display(structure)

## Series Lifespans

Each horizontal bar spans one series from its first to its last observation, coloured by volume. Late starts are new series with little history, which need a global model or hierarchical pooling. Early ends are discontinued series that should usually be excluded from training targets. The dashed line marks the forecast origin.

In [ ]:
show = life.head(40)
fig, ax = plt.subplots(figsize=(14, max(3.5, 0.26 * len(show) + 1.8)))
norm = plt.Normalize(np.log1p(show['total']).min(), np.log1p(show['total']).max())
for i, (name, r) in enumerate(show.iloc[::-1].iterrows()):
    ax.plot([r['start'], r['end']], [i, i], lw=6, solid_capstyle='butt', color=CMAP_SEQ(0.35 + 0.65 * norm(np.log1p(r['total']))))
    if r['n_obs'] < r['expected']:
        ax.text(r['end'], i, f'  {r["expected"] - r["n_obs"]} gaps', va='center', fontsize=7, color=ACCENT)
ax.set_yticks(range(len(show)))
ax.set_yticklabels([short(s, 24) for s in show.index[::-1]], fontsize=8)
ax.axvline(df[D].max(), color=ACCENT, ls='--', lw=1.2)
date_axis(ax)
extra = f', top 40 of {len(life)} by volume' if len(life) > 40 else ''
suptitle(fig, 'Series lifespans', f'frequency {FREQ}, darker = higher total volume{extra}')
save_fig(fig, 'series_lifespans')

#### Insights

> *Fill in after running.* What is the frequency and the horizon? How many series exist and how long are they? Are there new or discontinued series that need special handling?

---

# Completeness and Intermittency <a name="5"></a>

---

The long table is reshaped into a panel (timestamps x series) on the full regular grid. A missing timestamp *inside* a series' lifespan is a gap, while timestamps before its first or after its last observation are simply outside its life. Gaps must be filled before lags are computed, and whether they mean 'zero' (no sales were recorded) or 'unknown' (sensor outage) is a domain decision that the zero share helps make.

In [ ]:
panel = df.pivot(index=D, columns='series', values=Y).reindex(full_index)
panel = panel[life.index]
inside = pd.DataFrame({s: (full_index >= life.loc[s, 'start']) & (full_index <= life.loc[s, 'end']) for s in panel.columns},
                      index=full_index)
missing = panel.isna() & inside
STAT_SERIES = life.index[:CFG.MAX_SERIES_STAT].tolist()

def longest_run(mask):
    best = run = 0
    for v in mask:
        run = run + 1 if v else 0
        best = max(best, run)
    return best

comp = pd.DataFrame({
    'length': inside.sum(),
    'gaps': missing.sum(),
    'gap_%': missing.sum() / inside.sum() * 100,
    'longest_gap': [longest_run(missing[s].values) for s in panel.columns],
    'zero_%': (panel == 0).sum() / panel.notna().sum() * 100,
    'negative': (panel < 0).sum(),
})
save_table(comp, 'completeness')
display(comp.describe().T.round(2))

show = panel.columns[:40]
state = np.where(~inside[show], 0, np.where(missing[show], 1, np.where(panel[show] == 0, 2, 3))).T
step = max(1, len(full_index) // 400)
state = state[:, ::step]
fig, ax = plt.subplots(figsize=(15, max(3, 0.22 * len(show) + 1.8)))
cmap = mpl.colors.ListedColormap(['#F7F9FC', ACCENT, '#E9C46A', PRIMARY])
ax.imshow(state, aspect='auto', cmap=cmap, vmin=0, vmax=3, interpolation='nearest',
          extent=[mdates.date2num(full_index[0]), mdates.date2num(full_index[-1]), len(show), 0])
ax.set_yticks(np.arange(len(show)) + 0.5)
ax.set_yticklabels([short(s, 22) for s in show], fontsize=7)
ax.xaxis_date()
date_axis(ax)
ax.grid(False)
handles = [mpl.patches.Patch(color=c, label=l) for c, l in zip(['#F7F9FC', ACCENT, '#E9C46A', PRIMARY], ['outside lifespan', 'missing', 'zero', 'positive'])]
ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)
suptitle(fig, 'Data presence map', f'{int(missing.values.sum()):,} gaps inside lifespans ({missing.values.sum() / inside.values.sum():.2%}); top {len(show)} series by volume')
save_fig(fig, 'presence_map')

## Intermittency

Demand classification by Syntetos and Boylan (2005) uses the average demand interval (ADI, mean number of periods between non-zero values) and the squared coefficient of variation of non-zero sizes (CV squared). With cut-offs of 1.32 and 0.49 it separates smooth series (standard models work), erratic (volatile sizes), intermittent (many zeros, stable sizes, suited to Croston or TSB), and lumpy (both, the hardest).

In [ ]:
def adi_cv2(x):
    x = x[~np.isnan(x)]
    nz = np.flatnonzero(x > 0)
    if len(nz) < 2:
        return np.nan, np.nan
    sizes = x[nz]
    return len(x) / len(nz), (sizes.std() / sizes.mean()) ** 2

rows = []
for s in STAT_SERIES:
    a, c = adi_cv2(panel[s][inside[s]].values)
    cls = np.nan if np.isnan(a) else ('smooth' if a < 1.32 and c < 0.49 else 'erratic' if a < 1.32 else 'intermittent' if c < 0.49 else 'lumpy')
    rows.append({'series': s, 'adi': a, 'cv2': c, 'demand_class': cls})
intermit = pd.DataFrame(rows).set_index('series')
comp = comp.join(intermit)
save_table(intermit, 'intermittency')
class_counts = intermit['demand_class'].value_counts()
print('Demand classes:', class_counts.to_dict())

cls_color = {'smooth': PALETTE[2], 'erratic': PALETTE[3], 'intermittent': PRIMARY, 'lumpy': ACCENT}
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), gridspec_kw={'width_ratios': [1.5, 1]})
ax = axes[0]
for cl, g in intermit.dropna().groupby('demand_class'):
    ax.scatter(g['adi'], g['cv2'], s=40, color=cls_color[cl], alpha=0.8, label=f'{cl} ({len(g)})', edgecolor='white')
ax.axvline(1.32, color=MUTED, ls='--', lw=1)
ax.axhline(0.49, color=MUTED, ls='--', lw=1)
ax.set_xscale('log')
ax.set_xlabel('ADI (average demand interval, log)')
ax.set_ylabel('CV squared of non-zero sizes')
ax.legend()
ax.set_title('Syntetos-Boylan demand classification', loc='left')
cc = class_counts.reindex(['smooth', 'erratic', 'intermittent', 'lumpy']).fillna(0)
axes[1].bar(cc.index, cc.values, color=[cls_color[c] for c in cc.index])
bar_labels(axes[1])
axes[1].set_title('Series per class', loc='left')
suptitle(fig, 'Intermittency', 'thresholds ADI = 1.32, CV^2 = 0.49')
save_fig(fig, 'intermittency')

#### Insights

> *Fill in after running.* How many gaps exist and how long are they? Should they be filled with zero or interpolated? What share of series is intermittent or lumpy and therefore needs zero-aware models (Croston, TSB, or Tweedie / Poisson loss in gradient boosting)?

---

# Target Distribution and Transform <a name="6"></a>

---

Forecast models assume roughly constant variance. If the spread of a series grows with its level, a variance-stabilising transform is needed. The slope of log(std) against log(mean) across series measures this: near 1 means a log transform, near 0.5 a square root, and near 0 no transform (the Box-Cox lambda is approximately 1 minus the slope).

In [ ]:
stats_s = pd.DataFrame({'mean': panel[STAT_SERIES].mean(), 'std': panel[STAT_SERIES].std(),
                        'skew': panel[STAT_SERIES].skew(), 'min': panel[STAT_SERIES].min()})
ok = (stats_s['mean'] > 0) & (stats_s['std'] > 0)
slope = np.polyfit(np.log(stats_s.loc[ok, 'mean']), np.log(stats_s.loc[ok, 'std']), 1)[0] if ok.sum() >= 3 else np.nan
vals = df[Y].dropna()
if vals.min() < 0:
    TRANSFORM = 'none'
elif not np.isnan(slope) and slope > 0.75:
    TRANSFORM = 'log1p'
elif not np.isnan(slope) and slope > 0.3:
    TRANSFORM = 'sqrt'
else:
    TRANSFORM = 'log1p' if vals.skew() > 2 and vals.min() >= 0 else 'none'
if ok.sum() < 3:
    t_series = df.groupby(D)[Y].sum()
    roll = pd.DataFrame({'m': t_series.rolling(default_periods[0]).mean(), 's': t_series.rolling(default_periods[0]).std()}).dropna()
    roll = roll[(roll['m'] > 0) & (roll['s'] > 0)]
    if len(roll) > 10:
        slope = np.polyfit(np.log(roll['m']), np.log(roll['s']), 1)[0]
print(f'Variance-level slope: {slope:.2f} -> recommended transform: {TRANSFORM}')

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.histplot(vals, bins=60, color=PRIMARY, edgecolor='white', ax=axes[0])
axes[0].set_title(f'{Y} (skew {vals.skew():.2f}, zeros {(vals == 0).mean():.1%})', loc='left')
if vals.min() >= 0:
    sns.histplot(np.log1p(vals), bins=60, color=PALETTE[2], edgecolor='white', ax=axes[1])
    axes[1].set_title(f'log1p({Y}) (skew {np.log1p(vals).skew():.2f})', loc='left')
else:
    axes[1].set_visible(False)
if ok.sum() >= 3:
    axes[2].scatter(stats_s.loc[ok, 'mean'], stats_s.loc[ok, 'std'], s=35, color=ACCENT, alpha=0.75, edgecolor='white')
    xs = np.linspace(np.log(stats_s.loc[ok, 'mean']).min(), np.log(stats_s.loc[ok, 'mean']).max(), 50)
    b = np.polyfit(np.log(stats_s.loc[ok, 'mean']), np.log(stats_s.loc[ok, 'std']), 1)
    axes[2].plot(np.exp(xs), np.exp(b[1] + b[0] * xs), color=DARK, ls='--')
    axes[2].set_xscale('log')
    axes[2].set_yscale('log')
    axes[2].set_xlabel('series mean (log)')
    axes[2].set_ylabel('series std (log)')
    axes[2].set_title(f'Variance vs level across series (slope {slope:.2f})', loc='left')
else:
    axes[2].set_visible(False)
suptitle(fig, 'Target distribution and variance stabilisation', f'recommended transform: {TRANSFORM}')
save_fig(fig, 'target_distribution')

#### Insights

> *Fill in after running.* Is the target skewed, zero-heavy, or negative? Which transform is recommended and why? Remember to invert it on the forecasts and to evaluate on the original scale.

---

# Series Overview <a name="7"></a>

---

The aggregate series shows the overall level, trend, and calendar spikes, with a rolling mean over the main seasonal period. The number of active series is drawn on a second axis, because a rising total can simply reflect new series rather than real growth. Small multiples of the largest series then reveal how much individual series differ from the aggregate.

In [ ]:
total = panel.sum(axis=1, min_count=1)
active = inside.sum(axis=1)
m0 = default_periods[0]
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(total.index, total.values, color=SOFT, lw=0.8, label=Y)
ax.plot(total.index, total.rolling(m0, center=True).mean(), color=PRIMARY, lw=2, label=f'rolling mean ({m0})')
if len(total) > 6 * m0 * 4:
    ax.plot(total.index, total.rolling(m0 * 13 if FREQ_KEY == 'D' else m0 * 4, center=True).mean(), color=ACCENT, lw=2, label='long rolling mean')
ax2 = ax.twinx()
ax2.step(active.index, active.values, color=MUTED, lw=1.2, where='post', alpha=0.8)
ax2.set_ylabel('active series', color=MUTED)
ax2.grid(False)
ax.legend(loc='upper left')
date_axis(ax)
suptitle(fig, f'Aggregate {Y} over time', f'sum over {panel.shape[1]} series; grey step = number of active series')
save_fig(fig, 'aggregate_series')

top = panel.columns[:CFG.MAX_SERIES_PLOT]
ncols = 3
nrows = int(np.ceil(len(top) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 2.8 * nrows), squeeze=False, sharex=True)
for ax in axes.ravel()[len(top):]:
    ax.set_visible(False)
for i, (ax, s) in enumerate(zip(axes.ravel(), top)):
    x = panel[s]
    ax.plot(x.index, x.values, color=PALETTE[i % 10], lw=0.6, alpha=0.6)
    ax.plot(x.index, x.rolling(m0 * 4, center=True, min_periods=1).mean(), color=DARK, lw=1.4)
    ax.set_title(short(s, 30), loc='left', fontsize=10)
    date_axis(ax)
suptitle(fig, f'Top {len(top)} series by volume', f'dark line = rolling mean over {m0 * 4} periods')
save_fig(fig, 'top_series')

#### Insights

> *Fill in after running.* Is there an overall trend, and is it driven by growth or by new series? Are there visible spikes, drops, or regime changes? How heterogeneous are the series?

---

# Decomposition and Strength <a name="8"></a>

---

STL (Seasonal-Trend decomposition using Loess, Cleveland et al., 1990) splits a series into trend, seasonal, and remainder components and is robust to outliers. The strength of trend and seasonality (Wang, Smith and Hyndman, 2006) is then measured per series on a 0 to 1 scale. Strong seasonality favours seasonal models and seasonal lags, while strong trend calls for detrending, differencing, or trend-aware models.

$$F_T = \max\left(0, 1 - \frac{\text{Var}(R)}{\text{Var}(T + R)}\right), \qquad F_S = \max\left(0, 1 - \frac{\text{Var}(R)}{\text{Var}(S + R)}\right)$$

In [ ]:
def fill_series(x):
    x = x.copy()
    return x.interpolate(limit_direction='both') if x.notna().sum() > 1 else x.fillna(0)

def tf(x):
    return np.log1p(np.clip(x, 0, None)) if TRANSFORM == 'log1p' else np.sqrt(np.clip(x, 0, None)) if TRANSFORM == 'sqrt' else x

M = m0 if len(total) >= 3 * m0 else max(2, len(total) // 3)
agg = fill_series(total)
stl = STL(tf(agg), period=M, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
for ax, comp_, name, col in zip(axes, [tf(agg), stl.trend, stl.seasonal, stl.resid],
                                ['observed' + (f' ({TRANSFORM})' if TRANSFORM != 'none' else ''), 'trend', f'seasonal (period {M})', 'remainder'],
                                [SOFT, PRIMARY, PALETTE[2], ACCENT]):
    if name == 'remainder':
        ax.scatter(comp_.index, comp_.values, s=3, color=col)
        ax.axhline(0, color=DARK, lw=0.8)
    else:
        ax.plot(comp_.index, comp_.values, color=col, lw=1 if name.startswith('obs') else 1.6)
    ax.set_title(name, loc='left', fontsize=10)
date_axis(axes[-1])
suptitle(fig, 'STL decomposition of the aggregate series', f'robust STL, period {M}')
save_fig(fig, 'stl_aggregate')

def strength(x, period):
    x = fill_series(x.dropna())
    if len(x) < 2 * period + 1 or x.std() == 0:
        return np.nan, np.nan
    r = STL(tf(x), period=period, robust=True).fit()
    vr = np.var(r.resid)
    return max(0, 1 - vr / np.var(r.trend + r.resid)), max(0, 1 - vr / np.var(r.seasonal + r.resid))

st = pd.DataFrame([strength(panel[s][inside[s]], M) for s in STAT_SERIES[:300]], index=STAT_SERIES[:300], columns=['trend_strength', 'seasonal_strength'])
comp = comp.join(st)
save_table(st, 'strength')

fig, ax = plt.subplots(figsize=(9, 7))
for cl, g in comp.loc[st.index].groupby('demand_class'):
    ax.scatter(g['trend_strength'], g['seasonal_strength'], s=45, color=cls_color.get(cl, MUTED), alpha=0.8, edgecolor='white', label=cl)
ax.scatter([1 - np.var(stl.resid) / np.var(stl.trend + stl.resid)], [1 - np.var(stl.resid) / np.var(stl.seasonal + stl.resid)],
           marker='*', s=300, color=DARK, label='aggregate')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_xlabel('trend strength F_T')
ax.set_ylabel(f'seasonal strength F_S (period {M})')
ax.legend()
suptitle(fig, 'Trend vs seasonal strength per series', 'aggregates are usually smoother and more seasonal than individual series')
save_fig(fig, 'strength_map')
display(st.describe().T.round(3))

#### Insights

> *Fill in after running.* How strong are trend and seasonality, at the aggregate and per series? A big gap between them suggests forecasting at a higher level and reconciling, or using a global model that borrows strength across series.

---

# Seasonal Patterns <a name="9"></a>

---

Seasonal profiles show the typical shape within each cycle. Values are normalised by each series' mean before pooling, so large series do not dominate. The periodogram then finds the dominant cycle lengths directly from the data, which confirms the default periods or reveals unexpected ones (for example a 14-day payroll cycle).

In [ ]:
norm_long = (panel[STAT_SERIES] / panel[STAT_SERIES].mean()).stack().rename('y').reset_index()
norm_long.columns = ['date', 'series', 'y']
cal = {}
if FREQ_KEY in ('min', 'h'):
    cal['hour of day'] = norm_long['date'].dt.hour
if FREQ_KEY in ('min', 'h', 'D'):
    cal['day of week'] = norm_long['date'].dt.dayofweek
if FREQ_KEY == 'D':
    cal['day of month'] = norm_long['date'].dt.day
if (df[D].max() - df[D].min()).days > 400:
    cal['month'] = norm_long['date'].dt.month
    if FREQ_KEY in ('D', 'W'):
        cal['week of year'] = norm_long['date'].dt.isocalendar().week.astype(int)

fig, axes = plt.subplots(1, len(cal), figsize=(5.4 * len(cal), 4.4), squeeze=False)
PROFILES = {}
for i, (ax, (name, key)) in enumerate(zip(axes[0], cal.items())):
    prof = norm_long.groupby(key)['y'].agg(['mean', 'sem'])
    PROFILES[name] = prof['mean']
    ax.bar(prof.index, prof['mean'], yerr=1.96 * prof['sem'], color=PALETTE[i % 10], alpha=0.85,
           error_kw=dict(ecolor=DARK, lw=0.8, capsize=2))
    ax.axhline(1, color=DARK, ls='--', lw=1)
    lo = max(0, prof['mean'].min() - 0.15)
    ax.set_ylim(lo, prof['mean'].max() + 0.1)
    ax.set_title(f'{name} (range {prof["mean"].max() / max(prof["mean"].min(), 1e-9):.2f}x)', loc='left')
    if name == 'day of week':
        ax.set_xticks(range(7))
        ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
suptitle(fig, 'Seasonal profiles', 'mean of series-normalised target (1 = series average), whiskers = 95% CI')
save_fig(fig, 'seasonal_profiles')

## Year-over-Year View

Overlaying each year on a common calendar axis shows whether the seasonal shape is stable and whether it moves. Moving holidays such as Eid al-Fitr shift by about 11 days per year, which a fixed yearly seasonality cannot capture and which therefore need explicit holiday features. The heatmap shows the aggregate by year and month.

In [ ]:
if (df[D].max() - df[D].min()).days > 400:
    weekly_total = total.resample('W').sum(min_count=1) if FREQ_KEY in ('min', 'h', 'D') else total
    fig, axes = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={'width_ratios': [1.6, 1]})
    for i, (yr, g) in enumerate(weekly_total.groupby(weekly_total.index.year)):
        doy = g.index.dayofyear
        axes[0].plot(doy, g.values, color=PALETTE[i % 10], lw=1.8, label=str(yr), alpha=0.9)
    axes[0].set_xlabel('day of year')
    axes[0].legend(ncol=2, fontsize=8)
    axes[0].set_title('Weekly aggregate by year', loc='left')
    ym = total.groupby([total.index.year, total.index.month]).mean().unstack()
    sns.heatmap(ym, cmap=CMAP_HEAT, ax=axes[1], linewidths=0.4, linecolor='white', cbar_kws={'shrink': 0.7, 'label': f'mean {Y}'})
    axes[1].set_xlabel('month')
    axes[1].set_ylabel('year')
    axes[1].set_title('Mean aggregate by year and month', loc='left')
    suptitle(fig, 'Year-over-year seasonality', 'misaligned peaks across years point to moving holidays')
    save_fig(fig, 'year_over_year')
else:
    note('Less than about 13 months of data, so the year-over-year view is skipped.')

## Periodogram

The periodogram of the detrended aggregate measures how much variance each cycle length explains. The highest peaks (excluding very long cycles that are really trend) are the candidate seasonal periods. Final periods combine the frequency defaults with the detected peaks, keeping only periods that fit at least twice in the data.

In [ ]:
x = (stl.seasonal + stl.resid).values
freqs, power = signal.periodogram(x - x.mean(), detrend='linear')
mask = freqs > 0
periods_all, power = 1 / freqs[mask], power[mask]
keep = periods_all <= len(x) / 2
periods_all, power = periods_all[keep], power[keep]
peaks, _ = signal.find_peaks(power)
top_peaks = peaks[np.argsort(-power[peaks])][:6]
PEAKS = pd.DataFrame({'period': periods_all[top_peaks], 'power_share': power[top_peaks] / power.sum()}).sort_values('power_share', ascending=False)

if CFG.SEASONAL_PERIODS == 'auto':
    SEASONAL_PERIODS = [p for p in default_periods if len(total) >= 2 * p]
    for p in PEAKS['period'].head(3):
        rp = int(round(p))
        harmonic = any(abs(q / p - round(q / p)) < 0.05 for q in SEASONAL_PERIODS)
        if rp >= 2 and not harmonic and all(abs(rp - q) / q > 0.1 for q in SEASONAL_PERIODS) and PEAKS.set_index('period').loc[p, 'power_share'] > 0.05 and len(total) >= 2 * rp:
            SEASONAL_PERIODS.append(rp)
else:
    SEASONAL_PERIODS = list(CFG.SEASONAL_PERIODS)
SEASONAL_PERIODS = sorted(set(SEASONAL_PERIODS)) or [2]
save_table(PEAKS, 'periodogram_peaks', index=False)
display(PEAKS.round(4))
print(f'Seasonal periods: {SEASONAL_PERIODS}')

fig, ax = plt.subplots(figsize=(14, 4.6))
ax.semilogx(periods_all, power, color=PRIMARY, lw=1.2)
for p in top_peaks:
    ax.annotate(f'{periods_all[p]:.1f}', (periods_all[p], power[p]), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=8, color=ACCENT)
for p in SEASONAL_PERIODS:
    ax.axvline(p, color=ACCENT, ls='--', lw=1, alpha=0.7)
ax.set_xlabel('period (log scale, in time steps)')
ax.set_ylabel('power')
suptitle(fig, 'Periodogram of the detrended aggregate', f'dashed = selected seasonal periods {SEASONAL_PERIODS}')
save_fig(fig, 'periodogram')

#### Insights

> *Fill in after running.* Which calendar effects are strong (weekday, month, hour)? Which seasonal periods were selected, and does the year-over-year view show moving holidays that need explicit event features?

---

# Autocorrelation and Lags <a name="10"></a>

---

The autocorrelation function (ACF) shows how strongly a value relates to its own past at each lag, and the partial autocorrelation (PACF) removes the effect of intermediate lags. Significant lags become lag features for machine learning models, and the ACF averaged over series tells whether that pattern is shared. For direct multi-step forecasting, only lags of at least the horizon are usable without recursion, so both sets are reported.

In [ ]:
MAX_LAGS = int(CFG.MAX_LAGS) if CFG.MAX_LAGS != 'auto' else int(min(len(total) // 3, max(3 * max(SEASONAL_PERIODS[0], 2), 2 * HORIZON, 30)))
y_agg = tf(agg)
acf_vals = acf(y_agg, nlags=MAX_LAGS, fft=True)
pacf_vals = pacf(y_agg, nlags=min(MAX_LAGS, len(y_agg) // 2 - 1), method='ywm')
conf = 1.96 / np.sqrt(len(y_agg))

series_acf = []
for s in STAT_SERIES[:100]:
    xs = fill_series(panel[s][inside[s]])
    if len(xs) > MAX_LAGS + 10 and xs.std() > 0:
        series_acf.append(acf(tf(xs), nlags=MAX_LAGS, fft=True))
mean_acf = np.nanmean(series_acf, axis=0) if series_acf else acf_vals

lags = np.arange(len(acf_vals))
sig = [int(l) for l in np.argsort(-np.abs(mean_acf[1:]))[:15] + 1 if abs(mean_acf[l]) > conf]
LAGS_ALL = sorted(set(sig + [p for p in SEASONAL_PERIODS if p <= MAX_LAGS]))
LAGS_DIRECT = sorted({l for l in LAGS_ALL if l >= HORIZON} | {int(np.ceil(HORIZON / p) * p) for p in SEASONAL_PERIODS if p <= 4 * HORIZON} | {p for p in SEASONAL_PERIODS if p >= HORIZON})
base_w = sorted({w for w in [SEASONAL_PERIODS[0], 2 * SEASONAL_PERIODS[0], 4 * SEASONAL_PERIODS[0]] + ([SEASONAL_PERIODS[-1]] if SEASONAL_PERIODS[-1] > 4 * SEASONAL_PERIODS[0] else []) if w < len(total) // 2})
ROLL_WINDOWS = base_w
print(f'Significant lags (mean ACF): {sorted(sig)}')
print(f'Lag features (recursive / one-step): {LAGS_ALL}')
print(f'Lag features (direct, >= horizon {HORIZON}): {LAGS_DIRECT}')
print(f'Rolling windows: {ROLL_WINDOWS}')

fig, axes = plt.subplots(1, 2, figsize=(18, 4.8))
axes[0].vlines(lags[1:], 0, acf_vals[1:], color=PRIMARY, lw=1.2, label='aggregate')
axes[0].plot(lags[1:], mean_acf[1:], color=ACCENT, lw=1.8, label=f'mean over {len(series_acf)} series')
axes[0].fill_between(lags, -conf, conf, color=MUTED, alpha=0.2)
for p in SEASONAL_PERIODS:
    if p <= MAX_LAGS:
        axes[0].axvline(p, color=PALETTE[2], ls='--', lw=1)
axes[0].axvline(HORIZON, color=DARK, ls=':', lw=1.2, label=f'horizon {HORIZON}')
axes[0].legend(fontsize=8)
axes[0].set_title('Autocorrelation (ACF)', loc='left')
axes[0].set_xlabel('lag')
pl = np.arange(len(pacf_vals))
axes[1].vlines(pl[1:], 0, pacf_vals[1:], color=PALETTE[2], lw=1.4)
axes[1].fill_between(pl, -conf, conf, color=MUTED, alpha=0.2)
axes[1].set_title('Partial autocorrelation (PACF) of the aggregate', loc='left')
axes[1].set_xlabel('lag')
suptitle(fig, 'Autocorrelation structure', f'shaded = 95% band; green dashed = seasonal periods {SEASONAL_PERIODS}')
save_fig(fig, 'acf_pacf')

#### Insights

> *Fill in after running.* How persistent is the series (slow ACF decay means trend or non-stationarity)? Which lags are significant, and which are usable at the forecast horizon? A PACF that cuts off after p lags suggests an AR(p) component.

---

# Stationarity <a name="11"></a>

---

Two complementary tests are used because each has the opposite null hypothesis. The Augmented Dickey-Fuller (ADF) test assumes a unit root (non-stationary), and the KPSS test assumes stationarity. A series is called stationary when ADF rejects (p < 0.05) and KPSS does not (p >= 0.05). The tests are repeated after a log transform, a first difference, and a seasonal difference, which gives the differencing order an ARIMA model would need.

In [ ]:
def stationarity(x):
    x = pd.Series(x).dropna()
    if len(x) < 20 or x.std() == 0:
        return np.nan, np.nan
    adf_p = adfuller(x, autolag='AIC')[1]
    kpss_p = kpss(x, regression='c', nlags='auto')[1]
    return adf_p, kpss_p

m_s = SEASONAL_PERIODS[0]
variants = {'level': agg, f'{TRANSFORM}': tf(agg), 'first difference': tf(agg).diff(),
            f'seasonal difference ({m_s})': tf(agg).diff(m_s), f'seasonal + first difference': tf(agg).diff(m_s).diff()}
rows = []
for name, x in variants.items():
    a, k = stationarity(x)
    rows.append({'variant': name, 'adf_p': a, 'kpss_p': k, 'stationary': bool(a < 0.05 and k >= 0.05)})
stat_table = pd.DataFrame(rows).set_index('variant')
save_table(stat_table, 'stationarity_aggregate')
display(stat_table.round(4))

def n_diffs(x, max_d=2):
    for d in range(max_d + 1):
        a, k = stationarity(x)
        if a < 0.05 and k >= 0.05:
            return d
        x = pd.Series(x).diff().dropna()
    return max_d

rows = []
for s in STAT_SERIES[:60]:
    xs = tf(fill_series(panel[s][inside[s]]))
    a, k = stationarity(xs)
    rows.append({'series': s, 'adf_p': a, 'kpss_p': k, 'd': n_diffs(xs)})
stat_series = pd.DataFrame(rows).set_index('series')
save_table(stat_series, 'stationarity_series')
D_ORDER = int(stat_series['d'].median()) if len(stat_series) else n_diffs(tf(agg))
print(f'Series stationary at level: {(stat_series["d"] == 0).mean():.0%} | median differencing order d = {D_ORDER}')

fig, axes = plt.subplots(1, 2, figsize=(18, 4.6), gridspec_kw={'width_ratios': [2, 1]})
w = max(m_s * 4, 8)
axes[0].plot(agg.index, tf(agg).rolling(w).mean(), color=PRIMARY, lw=2, label=f'rolling mean ({w})')
ax2 = axes[0].twinx()
ax2.plot(agg.index, tf(agg).rolling(w).std(), color=ACCENT, lw=1.6, label=f'rolling std ({w})')
ax2.grid(False)
ax2.set_ylabel('rolling std', color=ACCENT)
axes[0].set_ylabel('rolling mean', color=PRIMARY)
axes[0].set_title('Rolling mean and std of the (transformed) aggregate', loc='left')
date_axis(axes[0])
dc = stat_series['d'].value_counts().sort_index()
axes[1].bar([f'd = {i}' for i in dc.index], dc.values, color=PALETTE[2])
bar_labels(axes[1])
axes[1].set_title('Differencing order needed per series', loc='left')
suptitle(fig, 'Stationarity', 'drifting mean or changing spread = non-stationary')
save_fig(fig, 'stationarity')

#### Insights

> *Fill in after running.* Is the aggregate stationary after the transform, a first difference, or a seasonal difference? For ARIMA this sets d and D. For tree models, a non-stationary level argues for predicting differences or ratios to a recent rolling mean rather than raw levels.

---

# Anomalies and Level Shifts <a name="12"></a>

---

Point anomalies are detected on robust STL remainders: a value is flagged when its remainder is more than `OUTLIER_Z` robust standard deviations (1.4826 times the median absolute deviation, floored at the remainder standard deviation) from the median. Intermittent and lumpy series are skipped, because their zeros make any deviation look extreme. Level shifts are detected by comparing the median of a window before and after each point. A candidate must also persist over a window three times longer, which filters out temporary spikes such as holidays. A persistent jump means older history may be misleading, so training could start after the break or include a regime indicator.

In [ ]:
rows, shifts = [], []
w_shift = max(4 * SEASONAL_PERIODS[0], 28 if FREQ_KEY == 'D' else 8)
smooth_series = [s for s in STAT_SERIES[:200] if comp.loc[s, 'demand_class'] in ('smooth', 'erratic')]
for s in smooth_series:
    raw_s = panel[s][inside[s]]
    xs = fill_series(raw_s)
    if len(xs) < 2 * M + 1 or xs.std() == 0:
        continue
    r = STL(tf(xs), period=M, robust=True).fit().resid
    mad = max(1.4826 * np.median(np.abs(r - np.median(r))), r.std(), 1e-9)
    z = (r - np.median(r)) / mad
    for d_, zz in z[(np.abs(z) > CFG.OUTLIER_Z) & raw_s.notna()].items():
        rows.append({'series': s, 'date': d_, 'value': raw_s[d_], 'robust_z': zz})
    lv = tf(xs).rolling(SEASONAL_PERIODS[0], center=True, min_periods=1).mean()
    before = lv.rolling(w_shift).median()
    after = lv[::-1].rolling(w_shift).median()[::-1]
    scale = max(1.4826 * (lv - lv.rolling(w_shift, center=True).median()).abs().median(), 0.25 * lv.std(), 1e-9)
    jump = ((after.shift(-1) - before) / scale).dropna()
    if len(jump):
        t = jump.abs().idxmax()
        b3 = lv[:t].tail(3 * w_shift).median()
        a3 = lv[t:].iloc[1:3 * w_shift + 1].median()
        shifts.append({'series': s, 'date': t, 'shift_in_mads': jump[t], 'persistent_shift': (a3 - b3) / scale})
anoms = pd.DataFrame(rows, columns=['series', 'date', 'value', 'robust_z'])
shift_df = pd.DataFrame(shifts, columns=['series', 'date', 'shift_in_mads', 'persistent_shift']).sort_values('shift_in_mads', key=np.abs, ascending=False)
LEVEL_SHIFTS = shift_df[(shift_df['shift_in_mads'].abs() > 4) & (shift_df['persistent_shift'].abs() > 6)]
save_table(anoms, 'anomalies', index=False)
save_table(shift_df, 'level_shift_candidates', index=False)
print(f'Point anomalies (|z| > {CFG.OUTLIER_Z}): {len(anoms):,} in {anoms["series"].nunique()} series')
print(f'Level shifts (> 4 MADs, persistent > 6): {len(LEVEL_SHIFTS)}')
display(LEVEL_SHIFTS.head(10))

pick = list(anoms['series'].value_counts().index[:2]) + list(LEVEL_SHIFTS['series'].head(2))
pick = list(dict.fromkeys(pick))[:4]
if pick:
    fig, axes = plt.subplots(len(pick), 1, figsize=(16, 2.9 * len(pick)), squeeze=False)
    for ax, s in zip(axes[:, 0], pick):
        x = panel[s]
        ax.plot(x.index, x.values, color=SOFT, lw=0.8)
        ax.plot(x.index, x.rolling(w_shift, center=True, min_periods=1).median(), color=PRIMARY, lw=1.6)
        a = anoms[anoms['series'] == s]
        ax.scatter(a['date'], a['value'], color=ACCENT, s=30, zorder=3, label='anomaly')
        sh = LEVEL_SHIFTS[LEVEL_SHIFTS['series'] == s]
        for d_ in sh['date']:
            ax.axvline(d_, color=DARK, ls='--', lw=1.4, label='level shift')
        ax.set_title(short(s, 40), loc='left', fontsize=10)
        ax.legend(loc='upper left', fontsize=8)
        date_axis(ax)
    suptitle(fig, 'Anomalies and level shifts', f'orange = STL remainder |z| > {CFG.OUTLIER_Z}, dashed = largest level shift')
    save_fig(fig, 'anomalies_level_shifts')

#### Insights

> *Fill in after running.* Are anomalies data errors (to clip or remove) or real events (to keep and explain with features)? Do level shifts suggest restricting the training window or adding a regime flag?

---

# Exogenous and Calendar Drivers <a name="13"></a>

---

Exogenous variables are only useful for forecasting if their future values are known (planned prices and promotions) or if they lead the target. Each covariate is therefore checked for presence in the test frame, its within-series correlation with the target at several lags, and, for binary flags, the uplift ratio. The largest positive residuals of the aggregate STL decomposition are listed as candidate special days, which usually reveal holidays that need explicit event features.

In [ ]:
EXOG_KNOWN = [c for c in EXOG if raw_test is not None and c in raw_test.columns]
rows = []
ccf_lags = list(range(0, min(3 * SEASONAL_PERIODS[0], 30) + 1))
CCF = {}
for c in EXOG:
    ex = df.pivot(index=D, columns='series', values=c).reindex(full_index)[STAT_SERIES[:100]]
    yy = tf(panel[STAT_SERIES[:100]])
    exd, yd = ex - ex.mean(), yy - yy.mean()
    ccf = [pd.concat([exd.shift(l).stack(), yd.stack()], axis=1).dropna().corr().iloc[0, 1] for l in ccf_lags]
    CCF[c] = ccf
    row = {'feature': c, 'known_in_future': c in EXOG_KNOWN, 'corr_lag0': ccf[0],
           'best_lag': ccf_lags[int(np.nanargmax(np.abs(ccf)))], 'best_corr': ccf[int(np.nanargmax(np.abs(ccf)))]}
    vals_c = df[c].dropna().unique()
    if set(np.unique(vals_c)) <= {0, 1}:
        g = df.groupby(['series', c])[Y].mean().unstack()
        row['uplift_ratio'] = float((g[1] / g[0]).replace([np.inf, -np.inf], np.nan).median()) if 1 in g and 0 in g else np.nan
    elif (df[c] > 0).all():
        el = []
        for s, g in df[df[Y] > 0].groupby('series'):
            if g[c].std() > 0 and len(g) > 30:
                el.append(np.polyfit(np.log(g[c]), np.log(g[Y]), 1)[0])
        row['elasticity'] = float(np.median(el)) if el else np.nan
    rows.append(row)
exog_table = pd.DataFrame(rows).set_index('feature') if rows else pd.DataFrame()
if not exog_table.empty:
    save_table(exog_table, 'exogenous')
    display(exog_table.round(3))

resid_agg = pd.Series(stl.resid, index=agg.index)
SPECIAL = resid_agg.nlargest(12)
special_tbl = pd.DataFrame({'date': SPECIAL.index.date, 'weekday': SPECIAL.index.day_name(), 'residual': SPECIAL.values})
save_table(special_tbl, 'special_day_candidates', index=False)

ncols = 2 if EXOG else 1
fig, axes = plt.subplots(1, ncols, figsize=(8.5 * ncols, 4.8), squeeze=False)
if EXOG:
    for i, (c, v) in enumerate(CCF.items()):
        axes[0, 0].plot(ccf_lags, v, marker='o', ms=3, lw=1.8, color=PALETTE[i % 10],
                        label=f'{c}' + (' (known future)' if c in EXOG_KNOWN else ' (past only)'))
    axes[0, 0].axhline(0, color=DARK, lw=0.8)
    axes[0, 0].set_xlabel('lag of covariate (periods)')
    axes[0, 0].set_ylabel('correlation with target')
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].set_title('Within-series cross-correlation', loc='left')
ax = axes[0, -1]
ax.plot(resid_agg.index, resid_agg.values, color=SOFT, lw=0.7)
ax.scatter(SPECIAL.index, SPECIAL.values, color=ACCENT, s=30, zorder=3)
for d_, v in SPECIAL.head(6).items():
    ax.annotate(d_.strftime('%Y-%m-%d'), (d_, v), fontsize=7, xytext=(3, 3), textcoords='offset points', color=DARK)
date_axis(ax)
ax.set_title('Aggregate STL remainder: candidate special days', loc='left')
suptitle(fig, 'Drivers of the target', 'use known-future covariates directly; past-only ones only as lags >= horizon')
save_fig(fig, 'drivers')
display(special_tbl)

#### Insights

> *Fill in after running.* Which covariates matter, and are they known for the forecast period? What are the promotion uplift and price elasticity? Do the candidate special days match known holidays (Lebaran, Christmas, payday), and should they become event features?

---

# Cross-Series Structure <a name="14"></a>

---

With many series, the key design question is local models (one per series) versus a single global model trained on all series. Highly correlated series and a heavy-tailed volume distribution both favour a global model, while the Pareto curve shows how much of the total the top series carry, which matters for volume-weighted metrics such as WAPE. When several ID columns exist, the volume by each level outlines the hierarchy.

In [ ]:
share = life['total'] / life['total'].sum()
cum = share.cumsum().values
n80 = int(np.searchsorted(cum, 0.8) + 1)
top = panel.columns[:min(20, panel.shape[1])]
resampled = panel[top].resample('W').sum(min_count=1) if FREQ_KEY in ('min', 'h', 'D') else panel[top]
corr = np.log1p(resampled.clip(lower=0)).diff().corr() if len(top) > 1 else pd.DataFrame()
MEAN_CORR = float(corr.values[np.triu_indices_from(corr.values, 1)].mean()) if len(top) > 2 else np.nan

n_extra = len(IDS) if len(IDS) > 1 else 0
fig, axes = plt.subplots(1, 2 + n_extra, figsize=(7 * (2 + n_extra), 5.6), squeeze=False)
ax = axes[0, 0]
ax.plot(np.arange(1, len(cum) + 1) / len(cum) * 100, cum * 100, color=PRIMARY, lw=2.4)
ax.axhline(80, color=MUTED, ls='--', lw=1)
ax.axvline(n80 / len(cum) * 100, color=ACCENT, ls='--', lw=1)
ax.set_xlabel('% of series (largest first)')
ax.set_ylabel('% of total volume')
ax.set_title(f'Pareto: {n80} series ({n80 / len(cum):.0%}) carry 80% of volume', loc='left')
if len(top) > 1:
    sns.heatmap(corr, cmap=CMAP_DIV, vmin=-1, vmax=1, center=0, ax=axes[0, 1], square=True,
                xticklabels=[short(c, 12) for c in corr.columns], yticklabels=[short(c, 12) for c in corr.index], cbar_kws={'shrink': 0.6})
    axes[0, 1].tick_params(labelsize=7)
    axes[0, 1].set_title(f'Correlation of weekly log-changes (mean {MEAN_CORR:.2f})', loc='left')
else:
    axes[0, 1].set_visible(False)
for j, col in enumerate(IDS if n_extra else []):
    lvl = raw.groupby(col)[Y].sum().sort_values()
    axes[0, 2 + j].barh([short(i, 18) for i in lvl.index[-20:]], lvl.values[-20:], color=PALETTE[(j + 2) % 10])
    axes[0, 2 + j].set_title(f'Volume by {col}', loc='left')
suptitle(fig, 'Cross-series structure', f'{panel.shape[1]} series')
save_fig(fig, 'cross_series')

#### Insights

> *Fill in after running.* Are series strongly co-moving (a shared driver that a global model can learn)? How concentrated is the volume? Should forecasting happen at the bottom level, at an aggregate level, or both with reconciliation?

---

# Baseline Forecastability <a name="15"></a>

---

Before modelling, simple benchmarks are backtested on the last `HORIZON` periods of every series: the naive forecast (repeat the last value), the seasonal naive (repeat the last season), and the moving average of the last season. They are scored with MASE (error scaled by the in-sample seasonal naive error, below 1 means better than seasonal naive) and WAPE (total absolute error over total actuals, robust to zeros). Spectral entropy measures how much structure a series has: lower entropy means a more forecastable series (Goerg, 2013).

$$\text{MASE} = \frac{\frac{1}{h}\sum_{t=1}^{h}|y_t - \hat{y}_t|}{\frac{1}{n-m}\sum_{t=m+1}^{n}|y_t - y_{t-m}|} \qquad \text{WAPE} = \frac{\sum |y_t - \hat{y}_t|}{\sum |y_t|}$$

In [ ]:
def spectral_entropy(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 16 or x.std() == 0:
        return np.nan
    _, p = signal.welch(x - x.mean(), nperseg=min(256, len(x)))
    p = p / p.sum()
    p = p[p > 0]
    return float(-(p * np.log(p)).sum() / np.log(len(p)))

H, m = HORIZON, SEASONAL_PERIODS[0]
rows, err_tot = [], {'naive': 0.0, 'seasonal_naive': 0.0, 'moving_average': 0.0}
act_tot = 0.0
for s in STAT_SERIES[:300]:
    xs = fill_series(panel[s][inside[s]]).values
    if len(xs) < H + 2 * m + 5:
        continue
    train_, test_ = xs[:-H], xs[-H:]
    scale = np.mean(np.abs(train_[m:] - train_[:-m])) + 1e-9
    fc = {'naive': np.repeat(train_[-1], H),
          'seasonal_naive': np.resize(train_[-m:], H),
          'moving_average': np.repeat(train_[-m:].mean(), H)}
    row = {'series': s, 'entropy': spectral_entropy(tf(train_))}
    for k, f in fc.items():
        e = np.abs(test_ - f)
        row[f'mase_{k}'] = e.mean() / scale
        err_tot[k] += e.sum()
    act_tot += np.abs(test_).sum()
    rows.append(row)
bench = pd.DataFrame(rows).set_index('series')
save_table(bench, 'baseline_backtest')
summary_bench = pd.DataFrame({'median_MASE': [bench[f'mase_{k}'].median() for k in err_tot],
                              'WAPE': [err_tot[k] / max(act_tot, 1e-9) for k in err_tot]}, index=list(err_tot))
BEST_BENCH = summary_bench['WAPE'].idxmin()
display(summary_bench.round(4))
print(f'Best naive benchmark: {BEST_BENCH} (WAPE {summary_bench.loc[BEST_BENCH, "WAPE"]:.3f}) -> the score any model must beat')

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
axes[0].bar(summary_bench.index, summary_bench['WAPE'], color=[ACCENT if k == BEST_BENCH else PRIMARY for k in summary_bench.index])
bar_labels(axes[0], '{:.3f}')
axes[0].set_title(f'WAPE of naive benchmarks on the last {H} periods', loc='left')
cls_map = comp['demand_class'].reindex(bench.index)
for cl in cls_map.dropna().unique():
    g = bench[cls_map == cl]
    axes[1].scatter(g['entropy'], g[f'mase_{BEST_BENCH}'], s=40, alpha=0.8, color=cls_color.get(cl, MUTED), edgecolor='white', label=cl)
axes[1].axhline(1, color=MUTED, ls='--', lw=1)
axes[1].set_xlabel('spectral entropy (lower = more forecastable)')
axes[1].set_ylabel(f'MASE of {BEST_BENCH}')
axes[1].legend(fontsize=8)
axes[1].set_title('Forecastability vs benchmark error per series', loc='left')
suptitle(fig, 'Baseline forecastability', f'{len(bench)} series backtested, horizon {H}, season {m}')
save_fig(fig, 'baseline_forecastability')

#### Insights

> *Fill in after running.* Which naive benchmark is best, and what WAPE must a model beat? Which series are inherently hard (high entropy), and should they get simpler models or be aggregated?

---

# Text Signal <a name="16"></a>

---

Text can only help a forecast through what was already published when the forecast is made. This section loads the documents (a text column of the main table, or a separate dated text file found in the dataset folder), embeds them with TF-IDF and SVD, aggregates them per series and period, and correlates each text feature at time t - lag with the detrended target at time t. Lags inside the forecast horizon are unusable, because that text does not exist yet at the forecast origin, so the recommended `text_lag` is the strongest lag at or beyond the horizon. The noise level 2/sqrt(n) separates a real lead from chance.

In [ ]:
TEXT_NAME = re.compile(r'(text|news|review|tweet|post|comment|article|doc|berita|ulasan|komentar)', re.I)

def word_mean(s):
    s = s.dropna().astype(str).head(2000)
    return float(s.str.split().str.len().mean()) if len(s) else 0.0

def find_text_file():
    """The configured text file, else a file in the dataset folder whose name looks like text (news.csv, reviews.csv)."""
    if CFG.TEXT_PATH:
        p = Path(CFG.TEXT_PATH)
        return p if p.exists() else None
    folder = Path(DATA_PATH).parent
    skip = {Path(DATA_PATH).name, Path(TEST_PATH).name if TEST_PATH else ''}
    cands = [f for f in sorted(folder.iterdir()) if f.is_file() and f.suffix.lower() in TABULAR_EXT
             and f.name not in skip and file_role(f) is None and TEXT_NAME.search(f.stem)]
    return cands[0] if cands else None

def load_documents(main, ids):
    """Documents as (date, text, id columns): from a separate text file, else from a text column of the main table."""
    tf = find_text_file()
    if tf is not None:
        t = load_table(tf)
        dcol = CFG.TEXT_DATE_COL if CFG.TEXT_DATE_COL != 'auto' else next(
            (c for c in t.columns if any(k in c.lower() for k in ('date', 'time', 'tanggal', 'published', 'created'))), t.columns[0])
        obj = [c for c in t.columns if c != dcol and not pd.api.types.is_numeric_dtype(t[c])]
        tcol = CFG.TEXT_COL if CFG.TEXT_COL != 'auto' else max(obj, key=lambda c: word_mean(t[c]))
        tid = [c for c in ids if c in t.columns] if CFG.TEXT_ID_COLS == 'auto' else list(CFG.TEXT_ID_COLS)
        source = f'file {tf.name}'
    else:
        t = main
        obj = [c for c in t.columns if c not in ids + ['series'] and not pd.api.types.is_numeric_dtype(t[c])
               and not pd.api.types.is_datetime64_any_dtype(t[c])]
        cands = [c for c in obj if word_mean(t[c]) >= 3]
        tcol = CFG.TEXT_COL if CFG.TEXT_COL != 'auto' else (max(cands, key=lambda c: word_mean(t[c])) if cands else None)
        if tcol is None or tcol not in t.columns:
            return None, {}
        dcol, tid, source = D, list(ids), f'column {tcol!r} of the main table'
    docs = pd.DataFrame({'date': pd.to_datetime(t[dcol], errors='coerce').values, 'text': t[tcol].fillna('').astype(str).values})
    for c in tid:
        docs[c] = t[c].astype(str).values
    docs = docs.dropna(subset=['date'])
    docs = docs[docs['text'].str.strip().str.len() > 0].reset_index(drop=True)
    info = {'text_source': source, 'text_col': tcol, 'text_date_col': dcol, 'text_id_cols': tid, 'text_n_docs': int(len(docs))}
    print(f'Text: {len(docs):,} documents from {source}, keyed by {tid or "date only (shared by every series)"}, '
          f'{docs["date"].min().date()} to {docs["date"].max().date()}')
    return docs, info

def to_bucket(dates, grid):
    """Map document dates to grid dates: the period that contains them (start- or end-labelled frequencies)."""
    grid = np.asarray(np.sort(np.unique(grid)), dtype='datetime64[ns]')
    d = np.asarray(dates, dtype='datetime64[ns]')
    start_labelled = str(FREQ).upper().startswith(('MS', 'QS', 'YS', 'AS', 'BMS', 'BQS'))
    idx = np.searchsorted(grid, d, side='right') - 1 if start_labelled else np.searchsorted(grid, d, side='left')
    ok = (idx >= 0) & (idx < len(grid))
    out = np.full(len(d), np.datetime64('NaT'), dtype='datetime64[ns]')
    out[ok] = grid[idx[ok]]
    return pd.to_datetime(out)

In [ ]:
TEXT_DECISIONS, TEXT_LAG_REC = {}, None
DOCS, TEXT_INFO = load_documents(raw, IDS)
if DOCS is None:
    note('No text column (strings of three or more words) and no text file (news.csv, reviews.csv, ...) were found. '
            'Set TEXT_COL or TEXT_*_PATH in Settings to add the text modality.')
else:
    hist_docs = DOCS[DOCS['date'] <= df[D].max()].reset_index(drop=True)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20_000, sublinear_tf=True)
    Xt = vec.fit_transform(hist_docs['text'])
    k = max(1, min(CFG.TEXT_DIM, Xt.shape[1] - 1, len(hist_docs) - 1))
    Z = TruncatedSVD(k, random_state=CFG.SEED).fit_transform(Xt)
    comp_cols = [f'c{j}' for j in range(k)]
    keys = [c for c in TEXT_INFO['text_id_cols'] if c in IDS]
    doc_feats = pd.DataFrame(Z, columns=comp_cols).assign(n_docs=1, words=hist_docs['text'].str.split().str.len().values,
                                                          bucket=to_bucket(hist_docs['date'], df[D].unique()))
    for c in keys:
        doc_feats[c] = hist_docs[c].values
    agg = doc_feats.dropna(subset=['bucket']).groupby(keys + ['bucket']).agg(
        n_docs=('n_docs', 'sum'), words=('words', 'mean'), **{c: (c, 'mean') for c in comp_cols}).reset_index()
    ids_of = raw.drop_duplicates('series').set_index('series')[keys] if keys else None
    tp = df[['series', D, Y]].copy()
    for c in keys:
        tp[c] = tp['series'].map(ids_of[c]).astype(str)
    tp = tp.merge(agg.rename(columns={'bucket': D}), on=keys + [D], how='left')
    tp['n_docs'] = tp['n_docs'].fillna(0)
    tp = tp.sort_values(['series', D]).reset_index(drop=True)

    g = tp.groupby('series')[Y]
    trend = g.transform(lambda s: s.shift(1).rolling(4 * M, min_periods=1).mean())
    spread = g.transform(lambda s: s.rolling(4 * M, min_periods=2).std()).replace(0, np.nan)
    tp['z'] = (tp[Y] - trend) / spread
    feats = ['n_docs'] + comp_cols
    max_lag = int(min(max(3 * HORIZON, 2 * M), 180))
    lags = list(range(0, max_lag + 1, max(1, max_lag // 60)))
    corr = pd.DataFrame(index=lags, columns=feats, dtype=float)
    for f in feats:
        fs = tp.groupby('series')[f]
        for L in lags:
            corr.loc[L, f] = tp['z'].corr(fs.shift(L))
    score = corr.abs().max(axis=1)
    usable = score[score.index >= HORIZON]
    n_obs = int(tp['z'].notna().sum())
    threshold = 2 / np.sqrt(max(n_obs, 1))
    TEXT_LAG_REC = int(usable.idxmax()) if len(usable) and usable.notna().any() else int(HORIZON)
    signal = float(usable.max()) if len(usable) else float('nan')
    useful = bool(signal > threshold)
    save_table(corr.round(4), 'text_lag_correlation')
    print(f'Strongest text-target link at a usable lag (>= horizon {HORIZON}): lag {TEXT_LAG_REC}, |r| = {signal:.3f} '
          f'(noise level {threshold:.3f}) -> text {"carries" if useful else "shows no clear"} lead signal')

    fig = plt.figure(figsize=(18, 9.5))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.15])
    ax0 = fig.add_subplot(gs[0, :])
    vol = tp.groupby(D)['n_docs'].sum()
    tot = tp.groupby(D)[Y].sum()
    ax0.bar(vol.index, vol.values, width=1.0 if FREQ_KEY in ('D', 'h', 'min') else 5, color=SOFT, label='documents')
    ax0.set_ylabel('documents per period')
    axb = ax0.twinx()
    axb.plot(tot.index, tot.rolling(max(1, M), min_periods=1).mean(), color=PRIMARY, lw=1.6, label=f'total {Y} (smoothed)')
    axb.set_ylabel(Y)
    axb.grid(False)
    ax0.set_title('Text volume and the target over time', loc='left')
    ax1 = fig.add_subplot(gs[1, 0])
    sns.heatmap(corr.abs().T, cmap=CMAP_SEQ, ax=ax1, cbar_kws={'label': '|r|'})
    ax1.set_xlabel('lag (periods): text at t - lag vs target at t')
    ax1.set_title('Lead correlation by text feature and lag', loc='left')
    ax2 = fig.add_subplot(gs[1, 1])
    ax2.plot(score.index, score.values, color=PRIMARY, lw=2)
    ax2.axvspan(0, HORIZON, color=MUTED, alpha=0.15, label='unusable: inside the horizon')
    ax2.axhline(threshold, color=MUTED, ls='--', lw=1, label='noise level 2/sqrt(n)')
    ax2.axvline(TEXT_LAG_REC, color=ACCENT, lw=2, label=f'recommended lag {TEXT_LAG_REC}')
    ax2.set_xlabel('lag (periods)')
    ax2.set_ylabel('max |r| over text features')
    ax2.legend(loc='upper right', fontsize=8)
    ax2.set_title('Best usable lead', loc='left')
    suptitle(fig, 'Text signal', f'{len(hist_docs):,} documents, TF-IDF -> {k} SVD components, target detrended per series')
    save_fig(fig, 'text_signal')
    TEXT_DECISIONS = {**TEXT_INFO, 'text_lag': TEXT_LAG_REC, 'text_signal': signal, 'text_noise_level': float(threshold),
                      'text_useful': useful, 'text_windows': sorted({1, int(M), int(4 * M)})}

#### Insights

> *Fill in after running.* Does any text feature lead the target beyond the horizon, and by how many periods? A lead well above the noise level justifies the text modality in the pipeline; a flat curve means the text describes the past rather than anticipating it.

---

# EDA Summary and Pipeline Decisions <a name="17"></a>

---

All findings are condensed into a markdown summary and a machine-readable `eda_decisions.json` that the forecasting pipeline reads for its `'auto'` settings: frequency, horizon, seasonal periods, transform, lag and rolling features, gap filling, backtesting folds, metric, and model families. Each recommendation follows directly from a section above.

In [ ]:
zero_share = float((df[Y] == 0).mean())
inter_share = float(intermit['demand_class'].isin(['intermittent', 'lumpy']).mean()) if len(intermit) else 0.0
gap_share = float(missing.values.sum() / inside.values.sum())
FILL = 'zero' if zero_share > 0.05 else 'interpolate'
metric = 'wape' if zero_share > 0.01 else 'smape'
min_len = int(life['n_obs'].median())
n_folds = int(max(1, min(5, (min_len - 2 * max(SEASONAL_PERIODS[0], 1) - H) // H)))
families = ['seasonal_naive']
if panel.shape[1] >= 10:
    families += ['lightgbm_global']
families += ['ets', 'arima'] if panel.shape[1] <= 200 else []
if inter_share > 0.2:
    families += ['croston_tsb']
decisions = {
    'source': str(DATA_PATH), 'date_col': D, 'target_col': Y, 'id_cols': IDS, 'freq': FREQ, 'freq_key': FREQ_KEY,
    'horizon': HORIZON, 'horizon_source': H_SOURCE, 'seasonal_periods': [int(p) for p in SEASONAL_PERIODS],
    'n_series': int(panel.shape[1]), 'median_length': min_len, 'transform': TRANSFORM, 'differencing_d': D_ORDER,
    'fill_missing': FILL, 'gap_share': gap_share, 'zero_share': zero_share, 'intermittent_share': inter_share,
    'lags_recursive': [int(l) for l in LAGS_ALL], 'lags_direct': [int(l) for l in LAGS_DIRECT], 'rolling_windows': [int(w) for w in ROLL_WINDOWS],
    'exog_known_future': EXOG_KNOWN, 'exog_past_only': [c for c in EXOG if c not in EXOG_KNOWN],
    'special_dates': [str(d_.date()) for d_ in SPECIAL.index], 'level_shifts': int(len(LEVEL_SHIFTS)),
    'anomalies': int(len(anoms)), 'cv': {'scheme': 'expanding_window', 'n_folds': n_folds, 'step': H, 'horizon': H},
    'metric': metric, 'secondary_metrics': ['mase', 'rmse'], 'best_naive': BEST_BENCH,
    'best_naive_wape': float(summary_bench.loc[BEST_BENCH, 'WAPE']), 'model_families': families,
    'global_model_recommended': bool(panel.shape[1] >= 10 or (not np.isnan(MEAN_CORR) and MEAN_CORR > 0.3)),
    **TEXT_DECISIONS,
}
(CFG.OUTPUT_DIR / 'eda_decisions.json').write_text(json.dumps(decisions, indent=2, default=str), encoding='utf-8')

lines = [f'# Forecasting EDA Summary: {DATA_PATH.name}', '', '## Structure',
         f'- {panel.shape[1]} series of `{Y}` at frequency **{FREQ}**, {df[D].min().date()} to {df[D].max().date()} ({len(full_index):,} steps)',
         f'- Horizon **{HORIZON}** ({H_SOURCE}); median series length {min_len}; {int(structure.loc["series starting late", "value"])} series start late',
         f'- Gaps {gap_share:.2%} of in-lifespan steps; zeros {zero_share:.1%}; intermittent or lumpy series {inter_share:.0%}', '',
         '## Patterns',
         f'- Transform: **{TRANSFORM}** (variance-level slope {slope:.2f}); differencing d = {D_ORDER}',
         f'- Seasonal periods: **{SEASONAL_PERIODS}**; median trend strength {st["trend_strength"].median():.2f}, seasonal strength {st["seasonal_strength"].median():.2f}',
         f'- Significant lags: {sorted(sig)}',
         f'- Candidate special days: ' + ', '.join(str(d_.date()) for d_ in SPECIAL.index[:6]),
         f'- Point anomalies: {len(anoms):,}; level shifts: {len(LEVEL_SHIFTS)}', '']
if not exog_table.empty:
    lines.append('## Drivers')
    for c, r in exog_table.iterrows():
        extra = f', uplift x{r["uplift_ratio"]:.2f}' if 'uplift_ratio' in r and not pd.isna(r.get('uplift_ratio')) else ''
        extra += f', elasticity {r["elasticity"]:.2f}' if 'elasticity' in r and not pd.isna(r.get('elasticity')) else ''
        lines.append(f'- `{c}`: corr {r["corr_lag0"]:+.2f} at lag 0, {"known in future" if r["known_in_future"] else "past only"}{extra}')
    lines.append('')
lines += ['## Benchmarks', f'- Best naive: **{BEST_BENCH}**, WAPE {summary_bench.loc[BEST_BENCH, "WAPE"]:.3f}, median MASE {summary_bench.loc[BEST_BENCH, "median_MASE"]:.3f}', '',
          '## Pipeline decisions',
          f'1. Fill gaps with **{FILL}**; apply **{TRANSFORM}** to the target and invert after forecasting',
          f'2. Features: lags {LAGS_DIRECT} (direct) or {LAGS_ALL[:10]} (recursive), rolling windows {ROLL_WINDOWS}, calendar parts, special-day flags'
          + (f', known covariates {EXOG_KNOWN}' if EXOG_KNOWN else ''),
          f'3. Validation: expanding-window backtest, {n_folds} folds of {H} steps',
          f'4. Metric: **{metric.upper()}** plus MASE (MAPE is unsafe with {zero_share:.0%} zeros)' if zero_share > 0 else f'4. Metric: **{metric.upper()}** plus MASE',
          f'5. Models to try: {families}' + (' (global model across series recommended)' if decisions['global_model_recommended'] else ''),
          '', '## Exported files', f'- Figures ({len(SAVED_FIGS)}): `{CFG.FIG_DIR}`', f'- Tables: `{CFG.TABLE_DIR}`', '- Decisions: `eda_decisions.json`']
summary_md = '\n'.join(lines)
(CFG.OUTPUT_DIR / 'eda_summary.md').write_text(summary_md, encoding='utf-8')
display(Markdown(summary_md))

The final cell lists every exported artefact.

In [ ]:
print(f'Figures in {CFG.FIG_DIR.resolve()}:')
for f in SAVED_FIGS:
    print('  ', f)
print(f'\nTables in {CFG.TABLE_DIR.resolve()}:')
for f in sorted(CFG.TABLE_DIR.glob('*.csv')):
    print('  ', f.name)
print(f'\nDecisions: {(CFG.OUTPUT_DIR / "eda_decisions.json").resolve()}')

#### Insights

> *Fill in after running.* Summarise the frequency and horizon, the dominant seasonality and events, the transform, the benchmark to beat, and the planned model family, each linked to the figure that justifies it.

---

# Part 2: Modelling Pipeline

---

*Everything above is the EDA; everything below is the pipeline. The pipeline reloads its own libraries and settings, inherits the data files resolved in Part 1, and reads `eda-output/eda_decisions.json` through `EDA_DIR`, so its target, task, columns to drop, validation scheme, metric, and transforms are the ones the EDA justified. Settings that are not on `'auto'` override the EDA, and the decision table records the source of every choice.*

*To use the pipeline settings, edit the `Settings` cell in the Pipeline Initialization section (models, tuning, ensembling). Data paths only need to be set once, in Part 1.*

## How to Configure

*Settings on `'auto'` are filled from `eda-output/eda_decisions.json` (written by `eda_forecasting.ipynb`) or computed from the data when the file is missing.*

| Setting | Purpose | Example |
|---|---|---|
| `*_PATH`, `TEST_*_PATH` | History file and future frame (dates plus known covariates) | `'train.csv'`, `'test.csv'` |
| `DATE_COL`, `TARGET_COL`, `ID_COLS` | Long-format columns | `'date'`, `'sales'`, `['store', 'item']` |
| `MODELS` | Any subset of `['lightgbm', 'xgboost', 'catboost']` | `['catboost']` |
| `STRATEGY` | `'direct'` (lags >= horizon, one pass) or `'recursive'` (short lags, step by step) | `'direct'` |
| `HORIZON`, `FREQ`, `SEASONAL_PERIODS`, `TRANSFORM`, `LAGS`, `ROLL_WINDOWS` | Forecast design, `'auto'` from EDA | `28`, `'D'`, `[7, 365]` |
| `OBJECTIVE`, `WEIGHT_BY_VOLUME` | Loss and sample weighting, `'auto'` from EDA | `'tweedie'`, `True` |
| `EVENT_DATES` | Known event dates (past and future), for example Eid al-Fitr | `['2025-03-31', ...]` |
| `N_FOLDS`, `METRIC` | Backtest folds and selection metric | `5`, `'wape'` |

*A model whose library is not installed is skipped, and the statistical baselines always run.*

## Metric

***WAPE** (weighted absolute percentage error) is the default when the series contain zeros, because it weights errors by volume and never divides by a zero actual. **MASE** scales the error by the in-sample seasonal naive error, so values below 1 beat seasonal naive. RMSE, sMAPE, and bias are reported alongside.*

$$\text{WAPE} = \frac{\sum_{i,t} |y_{i,t} - \hat{y}_{i,t}|}{\sum_{i,t} |y_{i,t}|} \qquad \text{MASE}_i = \frac{\frac{1}{h}\sum_{t} |y_{i,t} - \hat{y}_{i,t}|}{\frac{1}{n-m}\sum_{t=m+1}^{n} |y_{i,t} - y_{i,t-m}|}$$

---

# Pipeline Initialization <a name="18"></a>

---

## Environment Setup

The pipeline runs on CPU. On a panel of 30 series and about 1,300 days the full backtest with three boosting models takes a few minutes. Set `USE_GPU = True` on Kaggle to speed up XGBoost and CatBoost on large panels.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn statsmodels pyarrow openpyxl lightgbm xgboost catboost torch transformers sentencepiece

## Import Libraries

The boosting libraries are imported defensively, so a missing one is skipped and the statistical baselines still produce a complete forecast.

In [ ]:
import os
import re
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy.optimize import minimize
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import hashlib
from IPython.display import Markdown, display

AVAILABLE = {}
try:
    import lightgbm as lgb
    AVAILABLE['lightgbm'] = lgb.__version__
except Exception as e:
    AVAILABLE['lightgbm'] = False
    print(f'[warn] lightgbm unavailable: {str(e).strip().splitlines()[0][:100]}')
try:
    import xgboost as xgb
    AVAILABLE['xgboost'] = xgb.__version__
except Exception as e:
    AVAILABLE['xgboost'] = False
    print(f'[warn] xgboost unavailable: {str(e).strip().splitlines()[0][:100]}')
try:
    import torch
    from transformers import AutoModel, AutoTokenizer
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
    HAS_TRANSFORMERS = True
except Exception:
    HAS_TRANSFORMERS = False
try:
    import catboost
    from catboost import CatBoostRegressor, Pool
    AVAILABLE['catboost'] = catboost.__version__
except Exception as e:
    AVAILABLE['catboost'] = False
    print(f'[warn] catboost unavailable: {str(e).strip().splitlines()[0][:100]}')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
print('Libraries:', AVAILABLE)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## Settings

Paths, columns, the model list, and every forecasting design choice live here. `'auto'` values are resolved in section 4 from the EDA decisions file. `EVENT_DATES` defaults to the Eid al-Fitr (Lebaran) dates for Indonesia, the most influential retail event there. Replace or extend it with the events relevant to the competition, including future dates.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()
    _ON_COLAB  = Path('/content').exists() and not _ON_KAGGLE

    # 1) Data location: a file or a dataset folder (train / test / sample_submission are found by name)
    KAGGLE_PATH      = '/kaggle/input/<dataset-slug>'
    COLAB_PATH       = '/content/drive/MyDrive/<folder>'
    LOCAL_PATH       = 'data'
    TEST_KAGGLE_PATH = None
    TEST_COLAB_PATH  = None
    TEST_LOCAL_PATH  = None
    READ_KWARGS      = {}
    DEMO_IF_MISSING  = True
    EDA_DIR          = 'eda-output'

    # 2) Columns
    DATE_COL   = 'auto'
    TARGET_COL = 'auto'
    ID_COLS    = 'auto'
    EXOG_COLS  = 'auto'
    AGG        = 'sum'
    # 2b) Text modality: a text column of the main table, or a separate dated text file (news, reviews, posts)
    TEXT_KAGGLE_PATH = None
    TEXT_COLAB_PATH  = None
    TEXT_LOCAL_PATH  = None
    TEXT_COL         = 'auto'
    TEXT_DATE_COL    = 'auto'
    TEXT_ID_COLS     = 'auto'
    TEXT_DIM         = 8
    TEXT_LAG         = 'auto'
    TEXT_FEATURES    = 'auto'
    TEXT_WINDOWS     = 'auto'
    EMBED_MODEL      = 'intfloat/multilingual-e5-base'
    EMBED_MAX_LENGTH = 256
    EMBED_BATCH      = 32
    TEXT_ABLATION    = True

    # 3) Forecast design ('auto' = from EDA decisions)
    FREQ             = 'auto'
    HORIZON          = 'auto'
    SEASONAL_PERIODS = 'auto'
    TRANSFORM        = 'auto'
    FILL_MISSING     = 'auto'
    STRATEGY         = 'direct'
    LAGS             = 'auto'
    ROLL_WINDOWS     = 'auto'
    EVENT_DATES      = ['2021-05-13', '2022-05-02', '2023-04-22', '2024-04-10', '2025-03-31', '2026-03-20', '2027-03-10']
    EVENT_WINDOW     = 14
    N_FOLDS          = 'auto'
    METRIC           = 'auto'
    NON_NEGATIVE     = 'auto'

    # 4) Models
    MODELS           = ['lightgbm', 'xgboost', 'catboost']
    BASELINES        = ['naive', 'seasonal_naive', 'moving_average', 'ets']
    ETS_MAX_SERIES   = 200
    OBJECTIVE        = 'auto'
    WEIGHT_BY_VOLUME = 'auto'
    N_ESTIMATORS     = 3000
    LEARNING_RATE    = 0.03
    EARLY_STOPPING   = 150
    REFIT_IN_FOLD    = True
    MODEL_PARAMS     = {'lightgbm': {}, 'xgboost': {}, 'catboost': {}}
    USE_GPU          = False
    ENSEMBLE         = True
    INTERVAL         = 0.8

    # 5) Outputs
    DATA_PATH    = KAGGLE_PATH if _ON_KAGGLE else COLAB_PATH if _ON_COLAB else LOCAL_PATH
    TEST_PATH    = TEST_KAGGLE_PATH if _ON_KAGGLE else TEST_COLAB_PATH if _ON_COLAB else TEST_LOCAL_PATH
    TEXT_PATH    = TEXT_KAGGLE_PATH if _ON_KAGGLE else TEXT_COLAB_PATH if _ON_COLAB else TEXT_LOCAL_PATH
    OUTPUT_DIR   = Path('/kaggle/working/pipeline-output') if _ON_KAGGLE else Path('pipeline-output')
    BASELINE_DIR = OUTPUT_DIR / 'baseline'
    FINAL_DIR    = OUTPUT_DIR / 'final'

CFG = Settings()
for d in (CFG.BASELINE_DIR / 'figures', CFG.FINAL_DIR / 'figures', CFG.FINAL_DIR / 'backtests'):
    d.mkdir(parents=True, exist_ok=True)
env = 'Kaggle' if CFG._ON_KAGGLE else 'Colab' if CFG._ON_COLAB else 'Local'
print(f'Environment : {env}')
print(f'Data path   : {CFG.DATA_PATH}')
print(f'Test path   : {CFG.TEST_PATH}')
print(f'Output dir  : {CFG.OUTPUT_DIR.resolve()}')
# reuse the data files resolved in Part 1, so the paths are edited in one place only
CFG.DATA_PATH = str(DATA_PATH)
CFG.TEST_PATH = str(TEST_PATH) if globals().get('TEST_PATH') else None
print(f'Pipeline data inherited from Part 1: {CFG.DATA_PATH} | test: {CFG.TEST_PATH}')

## Demo Data Generator

The same generator as the EDA notebook, used only when the configured file is missing and `DEMO_IF_MISSING` is true.

In [ ]:
def make_demo(train_path, test_path, seed=42):
    rng = np.random.RandomState(seed)
    dates = pd.date_range('2021-01-01', '2024-06-30', freq='D')
    future = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=28, freq='D')
    lebaran = pd.to_datetime(['2021-05-13', '2022-05-02', '2023-04-22', '2024-04-10'])
    all_dates = dates.append(future)
    t = np.arange(len(all_dates))
    weekly = np.array([0.9, 0.85, 0.9, 0.95, 1.1, 1.35, 1.25])[all_dates.dayofweek]
    yearly = 1 + 0.15 * np.sin(2 * np.pi * (all_dates.dayofyear - 80) / 365.25)
    days_to_eid = np.min(np.abs((all_dates.values[:, None] - lebaran.values[None, :]).astype('timedelta64[D]').astype(int)), axis=1)
    eid = 1 + 1.2 * np.exp(-days_to_eid / 4.0) * (days_to_eid <= 14)
    rows, news = [], []
    words = 'cuaca hari ini kota ramai jalan macet pasar harga stabil warga libur sekolah acara musik'.split()
    for s in range(1, 7):
        # store announcements lift sales 30 to 36 days later, longer than the 28-day horizon
        ann = np.where((rng.rand(len(t)) < 0.03) & (t < len(dates)))[0]
        boost = np.ones(len(t))
        for i in ann:
            boost[i + 30:i + 37] *= 1.7
            news.append({'date': all_dates[i], 'store': f'S{s}', 'headline': f'toko S{s} umumkan diskon besar dan promo spesial bulan depan'})
        for i in np.where(rng.rand(len(dates)) < 0.6)[0]:
            news.append({'date': dates[i], 'store': f'S{s}', 'headline': ' '.join(rng.choice(words, 7))})
        for it in range(1, 6):
            base = rng.uniform(20, 120) if it <= 3 else rng.uniform(0.3, 1.5)
            trend = 1 + rng.uniform(-0.1, 0.35) * t / len(t)
            price0 = rng.uniform(10, 60)
            price = price0 * (1 + 0.04 * np.sin(t / 90 + s)) * np.where(rng.rand(len(t)) < 0.03, 0.85, 1.0)
            promo = (rng.rand(len(t)) < 0.06).astype(int)
            lam = base * trend * weekly * yearly * eid * (1 + 0.5 * promo) * (price / price0) ** -1.5 * boost
            if s == 3 and it == 1:
                lam = lam * np.where(all_dates >= '2023-03-01', 1.6, 1.0)
            y = rng.poisson(lam * rng.gamma(20, 1 / 20, len(t))).astype(float)
            frame = pd.DataFrame({'date': all_dates, 'store': f'S{s}', 'item': f'I{it}', 'sales': y,
                                  'price': price.round(2), 'promo': promo})
            if s == 6:
                frame = frame[frame['date'] >= '2022-07-01']
            rows.append(frame)
    full = pd.concat(rows, ignore_index=True)
    train = full[full['date'] <= dates[-1]].copy()
    spikes = train.sample(12, random_state=seed).index
    train.loc[spikes, 'sales'] *= 8
    train = train.drop(train[(train['date'].between('2022-02-10', '2022-02-16')) & (train['store'] == 'S2')].index)
    train = train.drop(train.sample(frac=0.01, random_state=seed).index)
    test = full[full['date'] > dates[-1]].drop(columns='sales')
    Path(train_path).parent.mkdir(parents=True, exist_ok=True)
    train.to_csv(train_path, index=False)
    test.to_csv(test_path, index=False)
    pd.DataFrame(news).to_csv(Path(train_path).parent / 'news.csv', index=False)
    print(f'Demo data written: {train_path} ({len(train):,} rows), {test_path} ({len(test):,} rows)')

## Load Dataset

The history and the optional future frame are loaded with the same resolver as the other templates. Column roles are fixed in section 4, once the EDA decisions are known.

In [ ]:
TABULAR_EXT = ('.csv', '.tsv', '.txt', '.parquet', '.pq', '.feather', '.xlsx', '.xls', '.json', '.zip', '.gz')

def load_table(path, **kwargs):
    suffixes = [s.lower() for s in path.suffixes]
    if '.parquet' in suffixes or '.pq' in suffixes:
        return pd.read_parquet(path, **kwargs)
    if '.feather' in suffixes:
        return pd.read_feather(path, **kwargs)
    if '.xlsx' in suffixes or '.xls' in suffixes:
        return pd.read_excel(path, **kwargs)
    if '.json' in suffixes:
        return pd.read_json(path, **kwargs)
    if '.tsv' in suffixes:
        kwargs.setdefault('sep', '\t')
    return pd.read_csv(path, low_memory=False, **kwargs)

def file_role(f):
    """Classify a data file by its name: 'train', 'test', 'sample' (submission template), or None."""
    stem = f.name.lower().split('.')[0]
    if 'sample' in stem or 'submission' in stem:
        return 'sample'
    if stem.startswith('test'):
        return 'test'
    if stem.startswith('train'):
        return 'train'
    return None

SKIP_DIRS = {'eda-output', 'pipeline-output', 'catboost_info', '.ipynb_checkpoints', '.git', '.venv', 'site-packages'}
LOCAL_DATA_DIRS = ('data', 'input', 'dataset')

def discover_files(data_path, test_path):
    """Resolve the train, test, and sample-submission files from a file, a dataset folder, or the input root."""
    p = Path(data_path) if data_path else None
    explicit_test = Path(test_path) if test_path and Path(test_path).exists() else None
    if test_path and explicit_test is None:
        print(f'[warn] test path {test_path} not found; searching the dataset folder instead')
    if p is not None and p.is_file():
        train, files = p, [f for f in p.parent.iterdir() if f.is_file() and f.suffix.lower() in TABULAR_EXT]
    else:
        if p is not None and p.is_dir():
            roots = [p]
        elif CFG._ON_KAGGLE:
            roots = [Path('/kaggle/input')]
        elif CFG._ON_COLAB:
            roots = [Path('/content')]
        else:
            # local: data/, input/, dataset/ next to the notebook first, then the notebook folder itself
            roots = [Path(d) for d in LOCAL_DATA_DIRS] + [Path('.')]
        if p is not None and not p.is_dir():
            print(f'[warn] {p} not found; searching {[str(r) for r in roots if r.exists()]}')
        train, files = None, []
        for r in (r for r in roots if r.exists()):
            files = [f for f in r.rglob('*') if f.is_file() and f.suffix.lower() in TABULAR_EXT
                     and not SKIP_DIRS & set(f.parts)]
            named = [f for f in files if file_role(f) == 'train']
            others = [f for f in files if file_role(f) is None]
            train = max(named or others, key=lambda f: f.stat().st_size, default=None)
            if train is not None:
                break
    if train is None:
        return None, explicit_test, None

    def pick(role):
        near = [f for f in files if file_role(f) == role and f.parent == train.parent]
        cands = near or [f for f in files if file_role(f) == role]
        exact = [f for f in cands if f.name.lower().split('.')[0] == role]
        return min(exact or cands, key=lambda f: len(f.name), default=None)

    test = explicit_test or pick('test')
    sample = pick('sample')
    for role, f in (('train', train), ('test', test), ('sample submission', sample)):
        print(f'{role:<18}: {f if f else "(none found)"}')
    return train, test, sample

DATA_PATH, TEST_PATH, SAMPLE_PATH = discover_files(CFG.DATA_PATH, CFG.TEST_PATH)
if DATA_PATH is None and CFG.DEMO_IF_MISSING:
    make_demo('demo-data/train.csv', 'demo-data/test.csv', CFG.SEED)
    DATA_PATH = Path('demo-data/train.csv')
    TEST_PATH = Path('demo-data/test.csv')
assert DATA_PATH is not None, f'{CFG.DATA_PATH} not found. Edit the path lines in Settings.'
if TEST_PATH is None:
    print('[warn] No test file configured or found in the dataset folder: the horizon comes from the frequency and submission.csv is aligned to no test frame.')
raw = load_table(DATA_PATH, **CFG.READ_KWARGS)
raw_test = load_table(TEST_PATH, **CFG.READ_KWARGS) if TEST_PATH else None
SAMPLE = load_table(SAMPLE_PATH) if SAMPLE_PATH else None
print(f'History: {raw.shape[0]:,} rows x {raw.shape[1]} cols' + (f' | future frame: {raw_test.shape[0]:,} rows' if raw_test is not None else ''))

---

# Toolkit <a name="19"></a>

---

The plotting identity matches the other templates. The metric functions work on long frames of actuals and forecasts, so every model, fold, and horizon step is scored identically. MASE uses a per-series scale computed only from data before each forecast origin.

In [ ]:
PRIMARY   = '#3D5A80'
ACCENT    = '#EE6C4D'
SOFT      = '#98C1D9'
DARK      = '#1B263B'
MUTED     = '#8D99AE'
PALETTE   = ['#3D5A80', '#EE6C4D', '#2A9D8F', '#E9C46A', '#9B5DE5',
             '#F15BB5', '#00BBF9', '#8AB17D', '#E76F51', '#264653']
CMAP_SEQ  = LinearSegmentedColormap.from_list('seq', ['#F7F9FC', '#98C1D9', '#3D5A80', '#1B263B'])

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#C9D1DB', 'axes.labelcolor': DARK, 'axes.titleweight': 'bold',
    'axes.titlesize': 12, 'axes.labelsize': 10, 'axes.titlecolor': DARK,
    'xtick.color': '#4A5568', 'ytick.color': '#4A5568',
    'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'legend.frameon': False, 'font.size': 10,
})

FIG_COUNTER = {'baseline': 0, 'final': 0}

def slug(text):
    return re.sub(r'[^a-z0-9]+', '_', str(text).lower()).strip('_')[:60]

def save_fig(fig, name, stage='final'):
    FIG_COUNTER[stage] += 1
    root = CFG.BASELINE_DIR if stage == 'baseline' else CFG.FINAL_DIR
    fig.savefig(root / 'figures' / f'{FIG_COUNTER[stage]:02d}_{slug(name)}.png', facecolor='white')
    plt.show()
    plt.close(fig)

def suptitle(fig, title, subtitle=None, layout=True):
    h = fig.get_figheight()
    if layout:
        fig.tight_layout(rect=(0, 0, 1, 1 - (0.85 if subtitle else 0.55) / h))
    fig.text(0.01, 1 - 0.1 / h, title, ha='left', va='top', fontsize=15, fontweight='bold', color=DARK)
    if subtitle:
        fig.text(0.01, 1 - 0.42 / h, subtitle, ha='left', va='top', fontsize=10, color=MUTED)

def date_axis(ax):
    loc = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))

def bar_labels(ax, fmt='{:,.0f}', size=8):
    for container in ax.containers:
        if hasattr(container, 'datavalues'):
            ax.bar_label(container, labels=[fmt.format(v) for v in container.datavalues], padding=3, fontsize=size, color='#4A5568')

def short(value, k=28):
    value = str(value)
    return value if len(value) <= k else value[:k - 3] + '...'

def note(msg):
    display(Markdown(f'> **Note:** {msg}'))

def to_json(obj, path):
    path.write_text(json.dumps(obj, indent=2, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o)), encoding='utf-8')

def metrics(frame):
    y, p = frame['y_true'].values, frame['y_pred'].values
    e = y - p
    denom = np.abs(y).sum()
    out = {'wape': np.abs(e).sum() / denom if denom > 0 else np.nan,
           'mae': np.abs(e).mean(), 'rmse': np.sqrt((e ** 2).mean()),
           'smape': np.mean(2 * np.abs(e) / np.clip(np.abs(y) + np.abs(p), 1e-9, None)),
           'bias': (p.sum() - y.sum()) / denom if denom > 0 else np.nan}
    if 'scale' in frame and 'series' in frame:
        out['mase'] = (frame.assign(ae=np.abs(e)).groupby('series')['ae'].mean() / frame.groupby('series')['scale'].first()).replace([np.inf], np.nan).mean()
    elif 'scale' in frame:
        out['mase'] = float((np.abs(e) / frame['scale'].values).mean())
    return {k: float(v) for k, v in out.items()}

---

# EDA-Driven Decisions <a name="20"></a>

---

Every design choice is resolved here, in priority order: an explicit value in `Settings`, then `eda_decisions.json`, then a value computed from the data. The decision table records the source of each choice, so the methodology chapter of the report can state exactly which EDA finding drove which modelling decision.

In [ ]:
eda_path = Path(CFG.EDA_DIR) / 'eda_decisions.json'
EDA = json.loads(eda_path.read_text()) if eda_path.exists() else {}
print(f'EDA decisions: {eda_path.resolve()} ({"found" if EDA else "not found, computing from data"})')
if EDA and Path(EDA.get('source', '')).name != DATA_PATH.name:
    print(f'[warn] EDA was run on {EDA.get("source")}, not {DATA_PATH.name}')
DECISIONS = []

def decide(name, user_value, eda_key, compute, why=''):
    if user_value != 'auto':
        value, source = user_value, 'user'
    elif eda_key and EDA.get(eda_key) is not None:
        value, source = EDA[eda_key], 'eda'
    else:
        value, source = compute(), 'computed'
    why = why(value) if callable(why) else why
    DECISIONS.append({'setting': name, 'value': str(value)[:80], 'source': source, 'why': why})
    return value

def guess_target(train, test):
    """Infer the target: the sample-submission target column, else the one train column missing from the test file."""
    if SAMPLE is not None and SAMPLE.shape[1] == 2 and SAMPLE.columns[1] in train.columns:
        return SAMPLE.columns[1]
    if test is not None:
        only_train = [c for c in train.columns if c not in test.columns]
        if len(only_train) == 1:
            return only_train[0]
    return None

def guess_date():
    for c in raw.columns:
        if 'date' in c.lower() or 'time' in c.lower() or c.lower() in ('ds', 'period', 'month', 'week'):
            return c
    return raw.columns[0]

D = decide('DATE_COL', CFG.DATE_COL, 'date_col', guess_date)
Y = decide('TARGET_COL', CFG.TARGET_COL, 'target_col',
           lambda: guess_target(raw, raw_test) or [c for c in raw.columns if pd.api.types.is_numeric_dtype(raw[c])][-1])
IDS = decide('ID_COLS', CFG.ID_COLS, 'id_cols', lambda: [c for c in raw.columns if c not in (D, Y) and not pd.api.types.is_numeric_dtype(raw[c]) and raw[c].dropna().astype(str).head(2000).str.split().str.len().mean() < 3])
IDS = [c for c in IDS if c in raw.columns]

def prep(frame):
    frame = frame.copy()
    frame[D] = pd.to_datetime(frame[D], errors='coerce')
    frame = frame.dropna(subset=[D])
    frame['series'] = frame[IDS].astype(str).agg(' | '.join, axis=1) if IDS else 'total'
    return frame

raw = prep(raw)
raw_test = prep(raw_test) if raw_test is not None else None
EXOG_ALL = [c for c in raw.columns if c not in IDS + [D, Y, 'series'] and pd.api.types.is_numeric_dtype(raw[c])] if CFG.EXOG_COLS == 'auto' else list(CFG.EXOG_COLS)
EXOG_KNOWN = decide('EXOG_KNOWN', 'auto' if CFG.EXOG_COLS == 'auto' else CFG.EXOG_COLS, 'exog_known_future',
                    lambda: [c for c in EXOG_ALL if raw_test is not None and c in raw_test.columns], 'covariates present in the future frame')
EXOG_KNOWN = [c for c in EXOG_KNOWN if c in raw.columns]
EXOG_PAST = [c for c in EXOG_ALL if c not in EXOG_KNOWN]
print(f'date={D!r} target={Y!r} ids={IDS} known covariates={EXOG_KNOWN} past-only={EXOG_PAST}')

## Frequency, Horizon, Seasonality, Transform

Fallback computations mirror the EDA: the frequency from the most common step, the horizon from the future frame, seasonal periods from the frequency, and the transform from the variance-to-level slope across series.

In [ ]:
OFFSET_DEFAULTS = {'min': ([60, 1440], 60), 'h': ([24, 168], 48), 'D': ([7, 365], 28), 'W': ([52], 13),
                   'MS': ([12], 12), 'ME': ([12], 12), 'QS': ([4], 8), 'YS': ([1], 3)}

def compute_freq():
    step = raw.sort_values(['series', D]).groupby('series')[D].diff().dropna().mode().iloc[0]
    days = step / pd.Timedelta(days=1)
    if days < 1:
        hours = max(1, int(round(days * 24)))
        return 'h' if hours == 1 else f'{hours}h'
    if days < 6:
        return 'D'
    if days < 25:
        return 'W-' + ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN'][raw[D].dt.dayofweek.mode().iloc[0]]
    if days < 80:
        return 'MS' if (raw[D].dt.day == 1).mean() > 0.8 else 'ME'
    return 'QS' if days < 300 else 'YS'

FREQ = decide('FREQ', CFG.FREQ, 'freq', compute_freq, 'most common step between timestamps')
FKEY = 'h' if FREQ.endswith('h') else 'W' if FREQ.startswith('W') else FREQ if FREQ in OFFSET_DEFAULTS else 'D'
periods_default, h_default = OFFSET_DEFAULTS.get(FKEY, ([7], 14))
HORIZON = int(decide('HORIZON', CFG.HORIZON, 'horizon',
                     lambda: int(raw_test.groupby('series')[D].nunique().median()) if raw_test is not None else h_default,
                     'steps in the future frame, else a frequency default'))
SEASONAL_PERIODS = [int(p) for p in decide('SEASONAL_PERIODS', CFG.SEASONAL_PERIODS, 'seasonal_periods', lambda: periods_default,
                                           'frequency defaults confirmed by the periodogram')]
M = SEASONAL_PERIODS[0]

def compute_transform():
    if raw[Y].min() < 0:
        return 'none'
    st = raw.groupby('series')[Y].agg(['mean', 'std'])
    st = st[(st['mean'] > 0) & (st['std'] > 0)]
    if len(st) >= 3:
        slope = np.polyfit(np.log(st['mean']), np.log(st['std']), 1)[0]
        return 'log1p' if slope > 0.75 else 'sqrt' if slope > 0.3 else 'none'
    return 'log1p' if raw[Y].skew() > 2 else 'none'

TRANSFORM = decide('TRANSFORM', CFG.TRANSFORM, 'transform', compute_transform, 'variance-to-level slope across series')
NON_NEG = bool(decide('NON_NEGATIVE', CFG.NON_NEGATIVE, None, lambda: bool(raw[Y].min() >= 0),
                     lambda v: 'target never negative, forecasts clipped at 0' if v else 'target takes negative values'))
ZERO_SHARE = float(EDA.get('zero_share', (raw[Y] == 0).mean()))
INTER_SHARE = float(EDA.get('intermittent_share', 0.0))
FILL = decide('FILL_MISSING', CFG.FILL_MISSING, 'fill_missing', lambda: 'zero' if ZERO_SHARE > 0.05 else 'interpolate',
              lambda v: 'zeros are common, so a missing record most likely means no activity' if v == 'zero'
                        else 'few zeros, so gaps are filled from neighbouring values')
METRIC = decide('METRIC', CFG.METRIC, 'metric', lambda: 'wape' if ZERO_SHARE > 0.01 else 'smape', 'zero share decides WAPE vs sMAPE')

## Features, Validation, and Loss

Lags come from the EDA autocorrelation analysis: with the direct strategy only lags of at least the horizon are allowed (all information must exist at the forecast origin), while the recursive strategy may use short lags. The objective follows the target treatment: squared error on a log-transformed target, Tweedie for raw zero-heavy targets, squared error otherwise. When the metric is volume-weighted (WAPE) and the target is log-transformed, rows are weighted by series volume so that the loss matches the metric.

In [ ]:
STRATEGY = CFG.STRATEGY
MIN_LAG = HORIZON if STRATEGY == 'direct' else 1

def compute_lags():
    base = [MIN_LAG, MIN_LAG + 1, MIN_LAG + 2] + [k * M for k in range(1, 9) if k * M >= MIN_LAG][:4]
    base += [p for p in SEASONAL_PERIODS[1:] if p >= MIN_LAG]
    return sorted(set(base))

eda_lags = 'lags_direct' if STRATEGY == 'direct' else 'lags_recursive'
LAGS = sorted({int(l) for l in decide('LAGS', CFG.LAGS, eda_lags, compute_lags, f'significant ACF lags usable with the {STRATEGY} strategy') if int(l) >= MIN_LAG})
ROLL_WINDOWS = sorted({int(w) for w in decide('ROLL_WINDOWS', CFG.ROLL_WINDOWS, 'rolling_windows',
                                               lambda: [M, 2 * M, 4 * M] + SEASONAL_PERIODS[1:], 'multiples of the main season')})
EVENTS = pd.to_datetime(CFG.EVENT_DATES) if CFG.EVENT_DATES else pd.DatetimeIndex([])
if EDA.get('special_dates'):
    print('EDA special-day candidates (check they are covered by EVENT_DATES):', EDA['special_dates'][:8])

def compute_folds():
    min_len = int(raw.groupby('series').size().median())
    return int(max(1, min(5, (min_len - 2 * M - HORIZON) // HORIZON)))

N_FOLDS = int(decide('N_FOLDS', CFG.N_FOLDS, None, lambda: int(EDA['cv']['n_folds']) if EDA.get('cv') else compute_folds(),
                     'expanding-window folds that fit in the median series length'))
OBJECTIVE = decide('OBJECTIVE', CFG.OBJECTIVE, None,
                   lambda: 'l2' if TRANSFORM in ('log1p', 'sqrt') else ('tweedie' if NON_NEG and (ZERO_SHARE > 0.1 or INTER_SHARE > 0.2) else 'l2'),
                   lambda v: 'raw zero-heavy target -> Tweedie' if v == 'tweedie' else 'transformed or continuous target -> L2')
WEIGHT_BY_VOLUME = bool(decide('WEIGHT_BY_VOLUME', CFG.WEIGHT_BY_VOLUME, None, lambda: bool(METRIC == 'wape' and TRANSFORM != 'none'),
                               lambda v: 'align a transformed-scale loss with the volume-weighted metric' if v
                                         else 'metric is not volume-weighted or the target is untransformed'))
MODELS = [m for m in CFG.MODELS if AVAILABLE.get(m)]
SKIPPED = [m for m in CFG.MODELS if m not in MODELS]
DECISIONS.append({'setting': 'MODELS', 'value': str(MODELS), 'source': 'user', 'why': f'skipped (not installed): {SKIPPED}' if SKIPPED else 'all requested models available'})
if EDA.get('model_families'):
    print('EDA model families:', EDA['model_families'])

decision_table = pd.DataFrame(DECISIONS).set_index('setting')
decision_table.to_csv(CFG.FINAL_DIR / 'decisions.csv')
display(decision_table)

#### Insights

> *Fill in after running.* Quote the decision table: which choices came from the EDA, which were overridden, and why. This is the bridge between the EDA chapter and the modelling chapter of the report.

---

# Data Preparation <a name="21"></a>

---

Each series is placed on a regular grid from its first observation to the end of the history, and the future frame is appended for the next `HORIZON` steps. Missing target values inside the lifespan are filled as decided (zero or interpolation), known covariates are forward and backward filled within each series, and the transform is applied. The untransformed target is kept for scoring.

In [ ]:
agg_map = {Y: CFG.AGG, **{c: 'mean' for c in EXOG_ALL}}
hist = raw.groupby(['series', D], as_index=False).agg(agg_map)
ids_map = raw.drop_duplicates('series').set_index('series')[IDS] if IDS else pd.DataFrame(index=hist['series'].unique())
LAST = hist[D].max()
FUTURE_DATES = pd.date_range(LAST, periods=HORIZON + 1, freq=FREQ)[1:]

frames = []
for s, g in hist.groupby('series'):
    grid = pd.date_range(g[D].min(), FUTURE_DATES[-1], freq=FREQ)
    f = g.set_index(D).reindex(grid)
    f.index.name = D
    f['series'] = s
    frames.append(f.reset_index())
data = pd.concat(frames, ignore_index=True)
data['is_future'] = data[D] > LAST
if raw_test is not None and EXOG_KNOWN:
    fut = raw_test.groupby(['series', D], as_index=False)[EXOG_KNOWN].mean()
    data = data.merge(fut, on=['series', D], how='left', suffixes=('', '_future'))
    for c in EXOG_KNOWN:
        data[c] = data[c].fillna(data[f'{c}_future'])
        data = data.drop(columns=f'{c}_future')

gap = data[Y].isna() & ~data['is_future']
if FILL == 'zero':
    data.loc[gap, Y] = 0.0
else:
    data[Y] = data.groupby('series')[Y].transform(lambda s: s.interpolate(limit_direction='both'))
    data.loc[data['is_future'], Y] = np.nan
for c in EXOG_ALL:
    data[c] = data.groupby('series')[c].transform(lambda s: s.ffill().bfill())
for c in IDS:
    data[c] = data['series'].map(ids_map[c])

def fwd(x):
    return np.log1p(np.clip(x, 0, None)) if TRANSFORM == 'log1p' else np.sqrt(np.clip(x, 0, None)) if TRANSFORM == 'sqrt' else x

def inv(x):
    x = np.expm1(x) if TRANSFORM == 'log1p' else np.square(np.clip(x, 0, None)) if TRANSFORM == 'sqrt' else x
    return np.clip(x, 0, None) if NON_NEG else x

data = data.sort_values(['series', D]).reset_index(drop=True)
data['y_raw'] = data[Y]
data['y'] = fwd(data[Y])
SERIES = data['series'].unique().tolist()
print(f'{len(SERIES)} series | {len(data):,} grid rows ({int(gap.sum()):,} gaps filled with {FILL}) | '
      f'history to {LAST.date()} | future {FUTURE_DATES[0].date()} to {FUTURE_DATES[-1].date()}')
display(data.head())

## Text Modality

Documents are taken from a separate dated text file in the dataset folder (`news.csv`, `reviews.csv`, ...) or from a text column of the main table. Each document is embedded (transformer embeddings reduced with PCA, or TF-IDF reduced with SVD, by `TEXT_FEATURES`), then documents are averaged per series and period (or per date when the text is not tied to a series) together with the document count and length. Every text feature is shifted by `TEXT_LAG` periods, never less than the horizon, and smoothed over `TEXT_WINDOWS`, so a forecast only uses text that existed at its origin. The lag comes from the EDA text signal analysis.

In [ ]:
TEXT_NAME = re.compile(r'(text|news|review|tweet|post|comment|article|doc|berita|ulasan|komentar)', re.I)

def word_mean(s):
    s = s.dropna().astype(str).head(2000)
    return float(s.str.split().str.len().mean()) if len(s) else 0.0

def find_text_file():
    """The configured text file, else a file in the dataset folder whose name looks like text (news.csv, reviews.csv)."""
    if CFG.TEXT_PATH:
        p = Path(CFG.TEXT_PATH)
        return p if p.exists() else None
    folder = Path(DATA_PATH).parent
    skip = {Path(DATA_PATH).name, Path(TEST_PATH).name if TEST_PATH else ''}
    cands = [f for f in sorted(folder.iterdir()) if f.is_file() and f.suffix.lower() in TABULAR_EXT
             and f.name not in skip and file_role(f) is None and TEXT_NAME.search(f.stem)]
    return cands[0] if cands else None

def load_documents(main, ids):
    """Documents as (date, text, id columns): from a separate text file, else from a text column of the main table."""
    tf = find_text_file()
    if tf is not None:
        t = load_table(tf)
        dcol = CFG.TEXT_DATE_COL if CFG.TEXT_DATE_COL != 'auto' else next(
            (c for c in t.columns if any(k in c.lower() for k in ('date', 'time', 'tanggal', 'published', 'created'))), t.columns[0])
        obj = [c for c in t.columns if c != dcol and not pd.api.types.is_numeric_dtype(t[c])]
        tcol = CFG.TEXT_COL if CFG.TEXT_COL != 'auto' else max(obj, key=lambda c: word_mean(t[c]))
        tid = [c for c in ids if c in t.columns] if CFG.TEXT_ID_COLS == 'auto' else list(CFG.TEXT_ID_COLS)
        source = f'file {tf.name}'
    else:
        t = main
        obj = [c for c in t.columns if c not in ids + ['series'] and not pd.api.types.is_numeric_dtype(t[c])
               and not pd.api.types.is_datetime64_any_dtype(t[c])]
        cands = [c for c in obj if word_mean(t[c]) >= 3]
        tcol = CFG.TEXT_COL if CFG.TEXT_COL != 'auto' else (max(cands, key=lambda c: word_mean(t[c])) if cands else None)
        if tcol is None or tcol not in t.columns:
            return None, {}
        dcol, tid, source = D, list(ids), f'column {tcol!r} of the main table'
    docs = pd.DataFrame({'date': pd.to_datetime(t[dcol], errors='coerce').values, 'text': t[tcol].fillna('').astype(str).values})
    for c in tid:
        docs[c] = t[c].astype(str).values
    docs = docs.dropna(subset=['date'])
    docs = docs[docs['text'].str.strip().str.len() > 0].reset_index(drop=True)
    info = {'text_source': source, 'text_col': tcol, 'text_date_col': dcol, 'text_id_cols': tid, 'text_n_docs': int(len(docs))}
    print(f'Text: {len(docs):,} documents from {source}, keyed by {tid or "date only (shared by every series)"}, '
          f'{docs["date"].min().date()} to {docs["date"].max().date()}')
    return docs, info

def to_bucket(dates, grid):
    """Map document dates to grid dates: the period that contains them (start- or end-labelled frequencies)."""
    grid = np.asarray(np.sort(np.unique(grid)), dtype='datetime64[ns]')
    d = np.asarray(dates, dtype='datetime64[ns]')
    start_labelled = str(FREQ).upper().startswith(('MS', 'QS', 'YS', 'AS', 'BMS', 'BQS'))
    idx = np.searchsorted(grid, d, side='right') - 1 if start_labelled else np.searchsorted(grid, d, side='left')
    ok = (idx >= 0) & (idx < len(grid))
    out = np.full(len(d), np.datetime64('NaT'), dtype='datetime64[ns]')
    out[ok] = grid[idx[ok]]
    return pd.to_datetime(out)

In [ ]:
DOCS, TEXT_INFO = load_documents(raw, IDS)
TEXT_FEATS = []
if DOCS is None:
    note('No text column or text file was found, so the pipeline runs as a plain forecaster. Set TEXT_COL or TEXT_*_PATH.')
else:
    TEXT_LAG = int(decide('TEXT_LAG', CFG.TEXT_LAG, 'text_lag', lambda: HORIZON,
                          lambda v: f'strongest usable text lead in the EDA (>= horizon {HORIZON})' if EDA.get('text_lag') else 'horizon (minimum leak-free lag)'))
    if TEXT_LAG < HORIZON:
        print(f'[warn] TEXT_LAG {TEXT_LAG} is inside the horizon and would leak future text; raised to {HORIZON}')
        TEXT_LAG = HORIZON
    TEXT_WINDOWS = sorted({int(w) for w in (CFG.TEXT_WINDOWS if CFG.TEXT_WINDOWS != 'auto' else EDA.get('text_windows', [1, M, 4 * M]))})
    hist_docs = DOCS[DOCS['date'] <= LAST].reset_index(drop=True)
    on_gpu = HAS_TRANSFORMERS and torch.cuda.is_available()
    mode = CFG.TEXT_FEATURES if CFG.TEXT_FEATURES != 'auto' else ('embed' if HAS_TRANSFORMERS and (on_gpu or len(hist_docs) <= 30_000) else 'tfidf')
    TEXT_MODE = decide('TEXT_FEATURES', CFG.TEXT_FEATURES, None, lambda: mode,
                       lambda v: f'{len(hist_docs):,} documents, GPU: {on_gpu}, transformers: {HAS_TRANSFORMERS}')
    k = max(1, min(CFG.TEXT_DIM, len(hist_docs) - 1))
    if TEXT_MODE == 'embed':
        enc = {}

        def encode(texts):
            if not enc:
                device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
                enc.update(tok=AutoTokenizer.from_pretrained(CFG.EMBED_MODEL),
                           model=AutoModel.from_pretrained(CFG.EMBED_MODEL).to(device).eval(), device=device)
            prefix = 'query: ' if 'e5' in CFG.EMBED_MODEL.lower() else ''
            out = []
            with torch.inference_mode():
                for i in range(0, len(texts), CFG.EMBED_BATCH):
                    b = enc['tok']([prefix + x for x in texts[i:i + CFG.EMBED_BATCH]], padding=True, truncation=True,
                                   max_length=CFG.EMBED_MAX_LENGTH, return_tensors='pt').to(enc['device'])
                    h = enc['model'](**b).last_hidden_state
                    m = b['attention_mask'].unsqueeze(-1).to(h.dtype)
                    out.append(torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1e-9), dim=1).float().cpu().numpy())
            return np.vstack(out)

        uniq = sorted(set(hist_docs['text']))
        digest = hashlib.md5((CFG.EMBED_MODEL + '\x00'.join(uniq)).encode('utf-8', 'ignore')).hexdigest()[:16]
        cache = CFG.OUTPUT_DIR / 'cache' / f'text_embed_{digest}.npy'
        if cache.exists():
            E = np.load(cache)
        else:
            t0 = time.time()
            E = encode(uniq)
            cache.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache, E)
            print(f'Embedded {len(uniq):,} unique documents with {CFG.EMBED_MODEL} in {time.time() - t0:.0f}s')
        pos = {t: i for i, t in enumerate(uniq)}
        Z = PCA(min(k, E.shape[1]), random_state=CFG.SEED).fit_transform(E[[pos[t] for t in hist_docs['text']]])
    else:
        Xt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20_000, sublinear_tf=True).fit_transform(hist_docs['text'])
        Z = TruncatedSVD(max(1, min(k, Xt.shape[1] - 1)), random_state=CFG.SEED).fit_transform(Xt)
    comp_cols = [f'c{j}' for j in range(Z.shape[1])]
    keys = [c for c in TEXT_INFO['text_id_cols'] if c in IDS]
    doc_feats = pd.DataFrame(Z, columns=comp_cols).assign(n_docs=1, words=hist_docs['text'].str.split().str.len().values,
                                                          bucket=to_bucket(hist_docs['date'], data[D].unique()))
    for c in keys:
        doc_feats[c] = hist_docs[c].values
    agg = doc_feats.dropna(subset=['bucket']).groupby(keys + ['bucket']).agg(
        n_docs=('n_docs', 'sum'), words=('words', 'mean'), **{c: (c, 'mean') for c in comp_cols}).reset_index()
    tf_ = data[['series', D] + keys].copy()
    for c in keys:
        tf_[c] = tf_[c].astype(str)
    tf_ = tf_.merge(agg.rename(columns={'bucket': D}), on=keys + [D], how='left')
    tf_.index = data.index
    tf_['n_docs'] = tf_['n_docs'].fillna(0)
    grp = tf_.groupby('series')
    for f in ['n_docs', 'words'] + comp_cols:
        lagged = grp[f].shift(TEXT_LAG)
        for w in TEXT_WINDOWS:
            name = f'text_{f}_lag{TEXT_LAG}_w{w}'
            r = lagged.groupby(tf_['series']).rolling(w, min_periods=1)
            data[name] = (r.sum() if f == 'n_docs' else r.mean()).reset_index(level=0, drop=True)
            TEXT_FEATS.append(name)
    print(f'Text features: {len(TEXT_FEATS)} ({TEXT_MODE}, {len(comp_cols)} components, lag {TEXT_LAG}, windows {TEXT_WINDOWS}). '
          f'Every value uses text published at least {TEXT_LAG} periods before the target date.')
    display(data.loc[data['is_future'], ['series', D] + TEXT_FEATS[:4]].head())

---

# Feature Engineering <a name="22"></a>

---

Features are computed per series on the transformed target. Every target-derived feature is shifted by at least `MIN_LAG` steps (the horizon for the direct strategy), so a row never sees a value that would be unknown at the forecast origin. The feature families are as follows.

| Family | Features | Source of the design |
|---|---|---|
| Lags | `lag_k` for each k in `LAGS` | EDA ACF analysis |
| Rolling | mean and std over `ROLL_WINDOWS`, EWM (alpha 0.1, 0.3) | EDA seasonal periods |
| Same season | mean of the last values at multiples of the main period | EDA seasonal strength |
| Intermittency | zero share in recent windows, steps since last non-zero (only when zeros are common) | EDA demand classes |
| Calendar | hour, weekday, day of month, week, month, year, month start/end flags | EDA seasonal profiles |
| Events | days to next / since last event, in-window flag | `EVENT_DATES` + EDA special days |
| Covariates | known covariates as is plus relative to their recent mean; past-only ones lagged | EDA driver analysis |
| Identity | series id and id columns (categorical), series age | panel structure |
| Text | document count, length, and embedding components, lagged by `TEXT_LAG` and smoothed | EDA text signal |

In [ ]:
INTERMITTENT_FEATURES = ZERO_SHARE > 0.05 or INTER_SHARE > 0.1

def event_features(dates):
    out = pd.DataFrame(index=dates.index)
    if len(EVENTS) == 0:
        return out
    ev = EVENTS.sort_values().values
    d = dates.values.astype('datetime64[D]')
    pos = np.searchsorted(ev.astype('datetime64[D]'), d)
    nxt = np.where(pos < len(ev), (ev[np.minimum(pos, len(ev) - 1)].astype('datetime64[D]') - d).astype(int), 9999)
    prv = np.where(pos > 0, (d - ev[np.maximum(pos - 1, 0)].astype('datetime64[D]')).astype(int), 9999)
    out['days_to_event'] = np.clip(nxt, 0, 60)
    out['days_since_event'] = np.clip(prv, 0, 60)
    out['event_window'] = ((nxt <= CFG.EVENT_WINDOW) | (prv <= CFG.EVENT_WINDOW)).astype(int)
    return out

def build_features(frame):
    KEYS = frame['series'].values
    g = frame.groupby('series', sort=False)['y']
    feats = {}
    for L in LAGS:
        feats[f'lag_{L}'] = g.shift(L)
    base = g.shift(MIN_LAG)
    gb = base.groupby(KEYS)
    for w in ROLL_WINDOWS:
        r = gb.rolling(w, min_periods=max(1, w // 2))
        feats[f'roll_mean_{w}'] = r.mean().reset_index(level=0, drop=True)
        if w <= 4 * M:
            feats[f'roll_std_{w}'] = r.std().reset_index(level=0, drop=True)
    for a in (0.1, 0.3):
        feats[f'ewm_{a}'] = gb.transform(lambda s: s.ewm(alpha=a, ignore_na=True).mean())
    ks = [k * M for k in range(1, 9) if k * M >= MIN_LAG][:4]
    if M > 1 and ks:
        feats['same_season_mean'] = pd.concat([g.shift(k) for k in ks], axis=1).mean(axis=1)
    if INTERMITTENT_FEATURES:
        nz = (base > 0).astype(float).where(base.notna())
        for w in sorted({M, 4 * M}):
            feats[f'nonzero_share_{w}'] = nz.groupby(KEYS).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        idx = pd.Series(np.where(base > 0, np.arange(len(frame)), np.nan), index=frame.index)
        feats['steps_since_nonzero'] = np.arange(len(frame)) - idx.groupby(KEYS).ffill().values
    dt = frame[D]
    if FKEY in ('min', 'h'):
        feats['hour'] = dt.dt.hour
    if FKEY in ('min', 'h', 'D'):
        feats['dow'] = dt.dt.dayofweek
        feats['dom'] = dt.dt.day
        feats['is_month_start'] = dt.dt.is_month_start.astype(int)
        feats['is_month_end'] = dt.dt.is_month_end.astype(int)
    if FKEY in ('min', 'h', 'D', 'W'):
        feats['week'] = dt.dt.isocalendar().week.astype(int).values
    feats['month'] = dt.dt.month
    feats['year'] = dt.dt.year
    feats['series_age'] = frame.groupby('series').cumcount()
    for c in EXOG_KNOWN:
        feats[c] = frame[c]
        if frame[c].nunique() > 2:
            past = frame.groupby('series')[c].shift(1)
            ref = past.groupby(KEYS).rolling(4 * M, min_periods=1).mean().reset_index(level=0, drop=True)
            feats[f'{c}_rel'] = frame[c] / ref.replace(0, np.nan)
    for c in EXOG_PAST:
        feats[f'{c}_lag'] = frame.groupby('series')[c].shift(MIN_LAG)
    for c in TEXT_FEATS:
        feats[c] = frame[c]
    out = pd.DataFrame(feats, index=frame.index)
    out = out.join(event_features(dt))
    for c in IDS + ['series']:
        out[f'id_{c}'] = pd.Categorical(frame[c].astype(str), categories=sorted(data[c].astype(str).unique())) if c in frame else np.nan
    return out

t0 = time.time()
FEAT = build_features(data)
FEATURES = FEAT.columns.tolist()
CAT_FEATURES = [c for c in FEATURES if c.startswith('id_')]
data['valid'] = FEAT[f'lag_{LAGS[0]}'].notna() if LAGS else True

def family(c):
    for k, v in (('text_', 'text'), ('lag_', 'lags'), ('roll_', 'rolling'), ('ewm_', 'rolling'), ('same_season', 'same season'), ('nonzero', 'intermittency'),
                 ('steps_since', 'intermittency'), ('event', 'events'), ('days_', 'events'), ('id_', 'identity'), ('series_age', 'identity')):
        if c.startswith(k):
            return v
    if c in EXOG_KNOWN or c.endswith('_rel') or c.endswith('_lag'):
        return 'covariates'
    return 'calendar'

inv_tbl = pd.Series([family(c) for c in FEATURES], index=FEATURES).value_counts().rename('features').to_frame()
print(f'{len(FEATURES)} features built in {time.time() - t0:.1f}s (strategy {STRATEGY}, min lag {MIN_LAG}, intermittency features: {INTERMITTENT_FEATURES})')
display(inv_tbl)
pd.DataFrame({'feature': FEATURES, 'family': [family(c) for c in FEATURES]}).to_csv(CFG.FINAL_DIR / 'feature_inventory.csv', index=False)

#### Insights

> *Fill in after running.* Which feature families were built, and which EDA finding motivated each one? Mention the event calendar used and the reason behind the minimum lag.

---

# Backtesting Design <a name="23"></a>

---

Forecasts are validated with an expanding-window backtest that mimics deployment: at each cutoff the model sees only the past and forecasts the next `HORIZON` steps. Folds are placed at the end of the history, one horizon apart, because the most recent behaviour is the most relevant. Only originally observed values are scored, never filled gaps. MASE scales are computed per series from data before each cutoff.

In [ ]:
data['observed'] = raw.groupby(['series', D])[Y].sum().reindex(pd.MultiIndex.from_frame(data[['series', D]])).notna().values
hist_dates = np.sort(data.loc[~data['is_future'], D].unique())
CUTOFFS = [pd.Timestamp(hist_dates[-(k * HORIZON) - 1]) for k in range(N_FOLDS, 0, -1)]
STEP = {d: i for i, d in enumerate(pd.date_range(hist_dates[0], FUTURE_DATES[-1], freq=FREQ))}
data['t_idx'] = data[D].map(STEP)
WIDE = data[~data['is_future']].pivot(index=D, columns='series', values='y_raw')

def scales(cutoff):
    h = WIDE.loc[:cutoff]
    s = (h - h.shift(M)).abs().mean()
    return s.where(s > 0, h.diff().abs().mean()).fillna(1.0).clip(lower=1e-6)

def val_mask(cutoff):
    t = STEP[pd.Timestamp(cutoff)]
    return (data['t_idx'] > t) & (data['t_idx'] <= t + HORIZON) & ~data['is_future']

fold_tbl = pd.DataFrame([{'fold': k + 1, 'cutoff': c.date(), 'train_rows': int(((data[D] <= c) & data['valid']).sum()),
                          'valid_rows': int((val_mask(c) & data['observed']).sum())} for k, c in enumerate(CUTOFFS)]).set_index('fold')
display(fold_tbl)

fig, ax = plt.subplots(figsize=(15, 0.5 * N_FOLDS + 1.8))
start = pd.Timestamp(hist_dates[0])
for k, c in enumerate(CUTOFFS):
    end = data.loc[val_mask(c), D].max()
    ax.barh(k, mdates.date2num(c) - mdates.date2num(start), left=mdates.date2num(start), color=SOFT, height=0.6)
    ax.barh(k, mdates.date2num(end) - mdates.date2num(c), left=mdates.date2num(c), color=ACCENT, height=0.6)
ax.barh(N_FOLDS, mdates.date2num(FUTURE_DATES[-1]) - mdates.date2num(LAST), left=mdates.date2num(LAST), color=DARK, height=0.6)
ax.set_yticks(range(N_FOLDS + 1))
ax.set_yticklabels([f'fold {k + 1}' for k in range(N_FOLDS)] + ['final forecast'])
ax.xaxis_date()
date_axis(ax)
handles = [mpl.patches.Patch(color=c, label=l) for c, l in ((SOFT, 'train'), (ACCENT, 'validate'), (DARK, 'forecast'))]
ax.legend(handles=handles, loc='upper left', ncol=3)
suptitle(fig, 'Expanding-window backtest', f'{N_FOLDS} folds of {HORIZON} steps at frequency {FREQ}')
save_fig(fig, 'backtest_design', 'final')

---

# Statistical Baselines <a name="24"></a>

---

Four per-series baselines run on every fold. Naive repeats the last value, seasonal naive repeats the last season, moving average forecasts the mean of the last season, and ETS (Holt-Winters exponential smoothing with damped trend and additive seasonality on the transformed target) is fitted when the number of series is manageable. They need no features and set the bar that the gradient boosting models must clear.

In [ ]:
def baseline_forecast(name, y, H):
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) == 0:
        return np.zeros(H)
    if name == 'naive' or len(y) < M:
        return np.repeat(y[-1], H)
    if name == 'seasonal_naive':
        return np.resize(y[-M:], H)
    if name == 'moving_average':
        return np.repeat(y[-M:].mean(), H)
    try:
        yy = fwd(y[-min(len(y), 20 * M + 2 * H):])
        seasonal = 'add' if M > 1 and len(yy) >= 2 * M + 2 else None
        fit = ExponentialSmoothing(yy, trend='add', damped_trend=True, seasonal=seasonal,
                                   seasonal_periods=M if seasonal else None, initialization_method='estimated').fit()
        return inv(fit.forecast(H))
    except Exception:
        return np.resize(y[-M:], H)

BASELINES = [b for b in CFG.BASELINES if b != 'ets' or len(SERIES) <= CFG.ETS_MAX_SERIES]
BT = {}
t0 = time.time()
for name in BASELINES:
    rows = []
    for k, c in enumerate(CUTOFFS):
        sc = scales(c)
        vm = data[val_mask(c)]
        for s, g in vm.groupby('series'):
            hist_s = WIDE[s].loc[:c].dropna()
            if len(hist_s) == 0:
                continue
            fc = baseline_forecast(name, hist_s.values, len(g))
            rows.append(pd.DataFrame({'series': s, D: g[D].values, 'fold': k + 1, 'h': np.arange(1, len(g) + 1),
                                      'y_true': g['y_raw'].values, 'y_pred': np.clip(fc, 0, None) if NON_NEG else fc,
                                      'scale': sc[s], 'observed': g['observed'].values}))
    bt = pd.concat(rows, ignore_index=True)
    BT[name] = bt[bt['observed']].drop(columns='observed').reset_index(drop=True)
    print(f'  [{name}] done ({time.time() - t0:.0f}s)')

def summarize(bt):
    per_fold = pd.DataFrame([metrics(g) for _, g in bt.groupby('fold')])
    return per_fold.mean().to_dict(), per_fold

rows = []
for name, bt in BT.items():
    mean, per_fold = summarize(bt)
    rows.append({'model': name, **mean, f'{METRIC}_std': per_fold[METRIC].std()})
base_lb = pd.DataFrame(rows).set_index('model').sort_values(METRIC)
BEST_BASELINE = base_lb.index[0]
base_lb.to_csv(CFG.BASELINE_DIR / 'baseline_leaderboard.csv')
for name, bt in BT.items():
    bt.to_csv(CFG.BASELINE_DIR / f'backtest_{name}.csv', index=False)
display(base_lb.round(4))
print(f'Best baseline: {BEST_BASELINE} ({METRIC} {base_lb.loc[BEST_BASELINE, METRIC]:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
axes[0].barh(base_lb.index[::-1], base_lb[METRIC][::-1], xerr=base_lb[f'{METRIC}_std'][::-1],
             color=[ACCENT if m == BEST_BASELINE else PRIMARY for m in base_lb.index[::-1]], error_kw=dict(ecolor=DARK, lw=1, capsize=3))
bar_labels(axes[0], '{:.3f}')
axes[0].set_title(f'Backtest {METRIC} (mean over folds, whiskers = std)', loc='left')
for i, (name, bt) in enumerate(BT.items()):
    byh = bt.groupby('h').apply(lambda g: metrics(g)[METRIC])
    axes[1].plot(byh.index, byh.values, marker='o', ms=3, lw=1.8, color=PALETTE[i % 10], label=name)
axes[1].set_xlabel('steps ahead')
axes[1].set_ylabel(METRIC)
axes[1].legend(fontsize=8)
axes[1].set_title('Error by forecast step', loc='left')
suptitle(fig, 'Statistical baselines', f'{N_FOLDS} folds x {HORIZON} steps x {len(SERIES)} series')
save_fig(fig, 'baseline_backtest', 'baseline')

#### Insights

> *Fill in after running.* Which baseline wins, and how does its error grow with the forecast step? A strong seasonal naive means seasonality dominates. A strong moving average means the series are noisy around a stable level.

---

# Gradient Boosting Forecasters <a name="25"></a>

---

A single global model per library is trained on all series. Inside each fold, the last `HORIZON` steps of the training window serve as an early-stopping set (never the validation window, which would leak), the best iteration is recorded, and the model is refitted on the full training window with that many trees. The direct strategy predicts the whole validation window in one pass. The recursive strategy predicts one step at a time and feeds each prediction back into the lag features. The objective and sample weights come from section 4.

In [ ]:
class GBM:
    def __init__(self, kind, n_iter=None):
        self.kind, self.n_iter = kind, n_iter

    def _params(self, n):
        tw = OBJECTIVE == 'tweedie'
        if self.kind == 'lightgbm':
            p = dict(n_estimators=n, learning_rate=CFG.LEARNING_RATE, num_leaves=63, min_child_samples=30,
                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=CFG.SEED,
                     n_jobs=-1, verbose=-1, objective='tweedie' if tw else 'regression')
            if tw:
                p['tweedie_variance_power'] = 1.3
        elif self.kind == 'xgboost':
            p = dict(n_estimators=n, learning_rate=CFG.LEARNING_RATE, max_depth=7, min_child_weight=5, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=1.0, tree_method='hist', enable_categorical=True, random_state=CFG.SEED,
                     n_jobs=-1, objective='reg:tweedie' if tw else 'reg:squarederror')
            if tw:
                p['tweedie_variance_power'] = 1.3
            if CFG.USE_GPU:
                p['device'] = 'cuda'
        else:
            p = dict(iterations=n, learning_rate=CFG.LEARNING_RATE * 2, depth=7, l2_leaf_reg=3.0, random_seed=CFG.SEED, verbose=0,
                     allow_writing_files=False, thread_count=-1, loss_function='Tweedie:variance_power=1.3' if tw else 'RMSE')
            if CFG.USE_GPU:
                p['task_type'] = 'GPU'
        p.update(CFG.MODEL_PARAMS.get(self.kind, {}))
        return p

    def _prep(self, X):
        if self.kind != 'catboost':
            return X
        X = X.copy()
        for c in CAT_FEATURES:
            X[c] = X[c].astype(str)
        return X

    def fit(self, X, y, w=None, X_es=None, y_es=None, w_es=None):
        n = self.n_iter or CFG.N_ESTIMATORS
        Xp = self._prep(X)
        es = X_es is not None and self.n_iter is None
        if self.kind == 'lightgbm':
            self.model = lgb.LGBMRegressor(**self._params(n))
            kw = dict(eval_set=[(X_es, y_es)], eval_sample_weight=[w_es], callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING, verbose=False)]) if es else {}
            self.model.fit(Xp, y, sample_weight=w, **kw)
            self.best = self.model.best_iteration_ if es and self.model.best_iteration_ else n
        elif self.kind == 'xgboost':
            p = self._params(n)
            if es:
                p['early_stopping_rounds'] = CFG.EARLY_STOPPING
            self.model = xgb.XGBRegressor(**p)
            self.model.fit(Xp, y, sample_weight=w, eval_set=[(X_es, y_es)] if es else None,
                           sample_weight_eval_set=[w_es] if es and w_es is not None else None, verbose=False)
            self.best = (self.model.best_iteration + 1) if es else n
        else:
            p = self._params(n)
            if es:
                p['early_stopping_rounds'] = CFG.EARLY_STOPPING
            self.model = CatBoostRegressor(**p)
            self.model.fit(Pool(Xp, y, cat_features=CAT_FEATURES, weight=w),
                           eval_set=Pool(self._prep(X_es), y_es, cat_features=CAT_FEATURES, weight=w_es) if es else None,
                           use_best_model=es)
            self.best = (self.model.get_best_iteration() + 1) if es else n
        return self

    def predict(self, X):
        return np.asarray(self.model.predict(self._prep(X)), dtype=float)

    def importance(self):
        if self.kind == 'lightgbm':
            v = self.model.booster_.feature_importance('gain')
        elif self.kind == 'xgboost':
            v = pd.Series(self.model.get_booster().get_score(importance_type='total_gain')).reindex(FEATURES).fillna(0).values
        else:
            v = self.model.get_feature_importance()
        s = pd.Series(v, index=FEATURES, dtype=float)
        return s / max(s.sum(), 1e-12)

    def contributions(self, X):
        if self.kind == 'lightgbm':
            return np.asarray(self.model.predict(X, pred_contrib=True))[:, :-1]
        if self.kind == 'xgboost':
            return self.model.get_booster().predict(xgb.DMatrix(X, enable_categorical=True), pred_contribs=True)[:, :-1]
        return self.model.get_feature_importance(Pool(self._prep(X), cat_features=CAT_FEATURES), type='ShapValues')[:, :-1]

def volume_weights(mask, cutoff):
    if not WEIGHT_BY_VOLUME:
        return None
    vol = WIDE.loc[:cutoff].tail(4 * max(M, HORIZON)).mean().clip(lower=1e-3)
    w = data.loc[mask, 'series'].map(vol).values
    return w / w.mean()

## Recursive Forecasting Helper

For the recursive strategy, the target after the cutoff is hidden, then each step is predicted and written back before the features of the next step are rebuilt. Only a recent tail of each series is used to rebuild features, which keeps the loop fast.

In [ ]:
TAIL = max(LAGS + ROLL_WINDOWS + [4 * M]) + 2 * HORIZON + 5

def recursive_predict(model, cutoff, dates):
    work = data[data['t_idx'] >= STEP[pd.Timestamp(cutoff)] - TAIL].copy()
    work.loc[work[D] > cutoff, 'y'] = np.nan
    out = {}
    for d in dates:
        sub = work[work[D] <= d]
        Xd = build_features(sub).loc[sub[D] == d, FEATURES]
        p = model.predict(Xd)
        work.loc[Xd.index, 'y'] = p
        out.update(dict(zip(Xd.index, p)))
    return pd.Series(out)

## Backtest Loop

Every active model runs over all folds. The backtest predictions, best iterations, gain importances, and the last fold's model (for interpretation) are stored.

In [ ]:
def es_split(cutoff):
    es_cut = pd.Timestamp(hist_dates[np.searchsorted(hist_dates, np.datetime64(cutoff)) - HORIZON])
    return es_cut

def train_mask(cutoff):
    return (data[D] <= cutoff) & data['valid'] & data['y'].notna() & ~data['is_future']

GBM_INFO = {}
for kind in MODELS:
    rows, iters, imps, t0 = [], [], [], time.time()
    for k, c in enumerate(CUTOFFS):
        feat = FEAT if STRATEGY == 'direct' else build_features(data.assign(y=data['y'].where(data[D] <= c)))
        tr = train_mask(c)
        es_cut = es_split(c)
        fit_m, es_m = tr & (data[D] <= es_cut), tr & (data[D] > es_cut)
        m = GBM(kind).fit(feat.loc[fit_m, FEATURES], data.loc[fit_m, 'y'], volume_weights(fit_m, es_cut),
                          feat.loc[es_m, FEATURES], data.loc[es_m, 'y'], volume_weights(es_m, es_cut))
        best = m.best
        if CFG.REFIT_IN_FOLD:
            m = GBM(kind, n_iter=max(best, 50)).fit(feat.loc[tr, FEATURES], data.loc[tr, 'y'], volume_weights(tr, c))
        vm = val_mask(c)
        if STRATEGY == 'direct':
            pred = pd.Series(m.predict(FEAT.loc[vm, FEATURES]), index=data.index[vm])
        else:
            pred = recursive_predict(m, c, sorted(data.loc[vm, D].unique()))
        sc = scales(c)
        v = data.loc[vm].copy()
        v['y_pred'] = inv(pred.reindex(v.index).values)
        v['h'] = v['t_idx'] - STEP[c]
        v = v[v['observed']]
        rows.append(pd.DataFrame({'series': v['series'].values, D: v[D].values, 'fold': k + 1, 'h': v['h'].values,
                                  'y_true': v['y_raw'].values, 'y_pred': v['y_pred'].values, 'scale': v['series'].map(sc).values}))
        iters.append(best)
        imps.append(m.importance())
        print(f'  [{kind}] fold {k + 1}/{N_FOLDS} cutoff {c.date()} best_iter {best} {METRIC} {metrics(rows[-1])[METRIC]:.4f} ({time.time() - t0:.0f}s)')
    BT[kind] = pd.concat(rows, ignore_index=True)
    GBM_INFO[kind] = {'iters': iters, 'importance': pd.concat(imps, axis=1).mean(axis=1), 'last_model': m, 'last_cutoff': c, 'time': time.time() - t0}
    BT[kind].to_csv(CFG.FINAL_DIR / 'backtests' / f'backtest_{kind}.csv', index=False)
if not MODELS:
    note('No boosting library is available, so the pipeline continues with the statistical baselines.')

#### Insights

> *Fill in after running.* How many trees did early stopping choose, and are they stable across folds? Does each boosting model beat the best baseline in every fold?

## Text Modality Ablation

The first boosting model is backtested again on the same folds without the text features. The per-fold comparison shows whether the text improves the forecast consistently or only by chance, and the importance share shows how much the model relies on it. The table is saved as `text_ablation.csv`.

In [ ]:
TEXT_ABLATION = None
if CFG.TEXT_ABLATION and TEXT_FEATS and MODELS:
    kind = MODELS[0]
    saved = FEATURES, CAT_FEATURES
    FEATURES = [f for f in saved[0] if f not in TEXT_FEATS]
    rows = []
    for k, c in enumerate(CUTOFFS):
        feat = FEAT if STRATEGY == 'direct' else build_features(data.assign(y=data['y'].where(data[D] <= c)))
        tr = train_mask(c)
        es_cut = es_split(c)
        fit_m, es_m = tr & (data[D] <= es_cut), tr & (data[D] > es_cut)
        m = GBM(kind).fit(feat.loc[fit_m, FEATURES], data.loc[fit_m, 'y'], volume_weights(fit_m, es_cut),
                          feat.loc[es_m, FEATURES], data.loc[es_m, 'y'], volume_weights(es_m, es_cut))
        if CFG.REFIT_IN_FOLD:
            m = GBM(kind, n_iter=max(m.best, 50)).fit(feat.loc[tr, FEATURES], data.loc[tr, 'y'], volume_weights(tr, c))
        vm = val_mask(c)
        pred = (pd.Series(m.predict(FEAT.loc[vm, FEATURES]), index=data.index[vm]) if STRATEGY == 'direct'
                else recursive_predict(m, c, sorted(data.loc[vm, D].unique())))
        v = data.loc[vm].copy()
        v['y_pred'] = inv(pred.reindex(v.index).values)
        v['h'] = v['t_idx'] - STEP[c]
        v = v[v['observed']]
        sc = scales(c)
        rows.append(pd.DataFrame({'series': v['series'].values, D: v[D].values, 'fold': k + 1, 'h': v['h'].values,
                                  'y_true': v['y_raw'].values, 'y_pred': v['y_pred'].values, 'scale': v['series'].map(sc).values}))
    FEATURES, CAT_FEATURES = saved
    no_text = pd.concat(rows, ignore_index=True)
    fold_with = [metrics(g)[METRIC] for _, g in BT[kind].groupby('fold')]
    fold_without = [metrics(g)[METRIC] for _, g in no_text.groupby('fold')]
    TEXT_ABLATION = pd.DataFrame({'with text': metrics(BT[kind]), 'without text': metrics(no_text)})
    gain = (TEXT_ABLATION.loc[METRIC, 'without text'] - TEXT_ABLATION.loc[METRIC, 'with text']) / TEXT_ABLATION.loc[METRIC, 'without text']
    TEXT_ABLATION.to_csv(CFG.FINAL_DIR / 'text_ablation.csv')
    display(TEXT_ABLATION.round(4))
    wins = int(np.sum(np.array(fold_with) < np.array(fold_without)))
    print(f'{kind}: {METRIC} {TEXT_ABLATION.loc[METRIC, "with text"]:.4f} with text vs {TEXT_ABLATION.loc[METRIC, "without text"]:.4f} '
          f'without ({gain:+.1%}); text wins {wins} of {len(fold_with)} folds')

    fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
    xs = np.arange(1, len(fold_with) + 1)
    axes[0].bar(xs - 0.18, fold_without, width=0.36, color=MUTED, label='without text')
    axes[0].bar(xs + 0.18, fold_with, width=0.36, color=ACCENT, label='with text')
    axes[0].set_xticks(xs)
    axes[0].set_xlabel('backtest fold')
    axes[0].set_ylabel(f'{METRIC} (lower is better)')
    axes[0].legend()
    axes[0].set_title('Backtest error per fold', loc='left')
    imp = GBM_INFO[kind]['importance']
    share = imp.groupby([family(c) for c in imp.index]).sum()
    share = (share / share.sum()).sort_values()
    axes[1].barh(share.index, share.values, color=[ACCENT if i == 'text' else PRIMARY for i in share.index])
    bar_labels(axes[1], '{:.1%}')
    axes[1].set_title(f'Importance share by feature family ({kind})', loc='left')
    suptitle(fig, 'Text modality ablation', f'{kind} backtest with and without the {len(TEXT_FEATS)} text features: {gain:+.1%} {METRIC}')
    save_fig(fig, 'text_ablation', 'final')
else:
    print('Text ablation skipped (no text features, no boosting model, or TEXT_ABLATION = False).')

#### Insights

> *Fill in after running.* How much does the text lower the backtest error, and in how many folds? Which share of the model's gain comes from the text family? Quote this as the headline multimodal result.

---

# Ensemble and Model Selection <a name="26"></a>

---

The boosting models and the best baseline are blended with non-negative weights summing to one, fitted on the backtest forecasts by minimising mean absolute error (the numerator of WAPE). Statistical and machine learning forecasts often fail in different situations, so the blend is usually more robust than either. The selected model is the candidate with the lowest mean backtest error across folds.

In [ ]:
KEY = ['series', D, 'fold']
cands = MODELS + [BEST_BASELINE]
WEIGHTS = None
if CFG.ENSEMBLE and len(cands) >= 2:
    merged = BT[cands[0]][KEY + ['h', 'y_true', 'scale']].copy()
    for c in cands:
        merged = merged.merge(BT[c][KEY + ['y_pred']].rename(columns={'y_pred': c}), on=KEY, how='inner')
    P = merged[cands].values
    yt = merged['y_true'].values
    n = len(cands)
    opt = minimize(lambda w: np.abs(yt - P @ w).mean(), np.full(n, 1 / n), method='SLSQP',
                   bounds=[(0, 1)] * n, constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1})
    WEIGHTS = {c: round(float(w), 4) for c, w in zip(cands, opt.x)}
    BT['ensemble'] = merged[KEY + ['h', 'y_true', 'scale']].assign(y_pred=P @ opt.x)
    BT['ensemble'].to_csv(CFG.FINAL_DIR / 'backtests' / 'backtest_ensemble.csv', index=False)
    print('Ensemble weights:', WEIGHTS)

rows = []
for name, bt in BT.items():
    mean, per_fold = summarize(bt)
    rows.append({'model': name, 'type': 'baseline' if name in BASELINES else 'ensemble' if name == 'ensemble' else 'gbm',
                 **mean, f'{METRIC}_std': per_fold[METRIC].std(), 'folds_beating_best_baseline':
                 int((per_fold[METRIC].values < summarize(BT[BEST_BASELINE])[1][METRIC].values).sum()) if name != BEST_BASELINE else np.nan})
LEADERBOARD = pd.DataFrame(rows).set_index('model').sort_values(METRIC)
BEST = LEADERBOARD.index[0]
LEADERBOARD.to_csv(CFG.FINAL_DIR / 'leaderboard.csv')
display(LEADERBOARD.round(4))
gain = (1 - LEADERBOARD.loc[BEST, METRIC] / LEADERBOARD.loc[BEST_BASELINE, METRIC]) * 100
print(f'Selected: {BEST} ({METRIC} {LEADERBOARD.loc[BEST, METRIC]:.4f}), {gain:+.1f}% vs best baseline {BEST_BASELINE}')

lb = LEADERBOARD.iloc[::-1]
fig, axes = plt.subplots(1, 2 if WEIGHTS else 1, figsize=(17 if WEIGHTS else 10, 0.5 * len(lb) + 2.2), squeeze=False)
colors = [ACCENT if m == BEST else MUTED if lb.loc[m, 'type'] == 'baseline' else PRIMARY for m in lb.index]
axes[0, 0].barh(lb.index, lb[METRIC], xerr=lb[f'{METRIC}_std'], color=colors, error_kw=dict(ecolor=DARK, lw=1, capsize=3))
bar_labels(axes[0, 0], '{:.3f}')
axes[0, 0].set_title(f'Backtest {METRIC} (mean over {N_FOLDS} folds)', loc='left')
if WEIGHTS:
    w = pd.Series(WEIGHTS).sort_values()
    axes[0, 1].barh(w.index, w.values, color=PALETTE[2])
    bar_labels(axes[0, 1], '{:.2f}')
    axes[0, 1].set_title('Ensemble weights', loc='left')
suptitle(fig, 'Forecast model leaderboard', f'selected: {BEST} (orange); baselines in grey')
save_fig(fig, 'leaderboard', 'final')

#### Insights

> *Fill in after running.* Which model is selected, by how much does it beat the best baseline, and in how many folds? How is the ensemble weight split between machine learning and statistical forecasts?

---

# Backtest Diagnostics <a name="27"></a>

---

Aggregate scores hide where forecasts fail. Error is broken down by forecast step (does accuracy decay with distance?), by series (which series are hard, and does the model beat the baseline on them?), and by the EDA demand class (smooth versus intermittent). Actual and forecast paths for the largest series on the last fold make the comparison concrete.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
for i, name in enumerate(LEADERBOARD.index):
    byh = BT[name].groupby('h').apply(lambda g: metrics(g)[METRIC])
    is_base = name in BASELINES
    axes[0].plot(byh.index, byh.values, lw=2.4 if name == BEST else 1.4, ls='--' if is_base else '-',
                 color=ACCENT if name == BEST else PALETTE[i % 10], label=name, alpha=0.9)
axes[0].set_xlabel('steps ahead')
axes[0].set_ylabel(METRIC)
axes[0].legend(fontsize=8)
axes[0].set_title('Error by forecast step (dashed = baselines)', loc='left')

per_s = pd.DataFrame({n: BT[n].groupby('series').apply(lambda g: metrics(g)[METRIC]) for n in (BEST, BEST_BASELINE)})
vol = WIDE.sum()
axes[1].scatter(per_s[BEST_BASELINE], per_s[BEST], s=20 + 300 * vol.reindex(per_s.index) / vol.max(), color=PRIMARY, alpha=0.7, edgecolor='white')
lim = [0, np.nanmax(per_s.values) * 1.05]
axes[1].plot(lim, lim, color=MUTED, ls='--')
axes[1].set_xlabel(f'{METRIC} of {BEST_BASELINE}')
axes[1].set_ylabel(f'{METRIC} of {BEST}')
share_better = (per_s[BEST] < per_s[BEST_BASELINE]).mean()
axes[1].set_title(f'Per series: {BEST} better on {share_better:.0%} (below diagonal)', loc='left')
suptitle(fig, 'Backtest diagnostics', 'bubble size = series volume')
save_fig(fig, 'backtest_diagnostics', 'final')
per_s.to_csv(CFG.FINAL_DIR / 'error_by_series.csv')

cls_path = Path(CFG.EDA_DIR) / 'tables' / 'intermittency.csv'
if cls_path.exists():
    cls = pd.read_csv(cls_path).set_index('series')['demand_class']
    by_cls = pd.DataFrame({n: BT[n].assign(c=BT[n]['series'].map(cls)).groupby('c').apply(lambda g: metrics(g)[METRIC]) for n in LEADERBOARD.index})
    by_cls.to_csv(CFG.FINAL_DIR / 'error_by_demand_class.csv')
    display(by_cls.round(3))

In [ ]:
last = BT[BEST][BT[BEST]['fold'] == N_FOLDS]
top = WIDE.sum().sort_values(ascending=False).index[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 7.5), squeeze=False)
c = CUTOFFS[-1]
for ax, s in zip(axes.ravel(), top):
    h = WIDE[s].loc[:c].tail(4 * HORIZON)
    ax.plot(h.index, h.values, color=MUTED, lw=1, label='history')
    g = last[last['series'] == s]
    ax.plot(g[D], g['y_true'], color=DARK, lw=1.4, label='actual')
    ax.plot(g[D], g['y_pred'], color=ACCENT, lw=2, label=BEST)
    b = BT[BEST_BASELINE][(BT[BEST_BASELINE]['fold'] == N_FOLDS) & (BT[BEST_BASELINE]['series'] == s)]
    ax.plot(b[D], b['y_pred'], color=PRIMARY, lw=1.4, ls='--', label=BEST_BASELINE)
    ax.axvline(c, color=MUTED, ls=':', lw=1)
    ax.set_title(short(s, 30), loc='left', fontsize=10)
    date_axis(ax)
axes[0, 0].legend(fontsize=8)
suptitle(fig, f'Last backtest fold: actual vs forecast', f'cutoff {c.date()}, top 6 series by volume')
save_fig(fig, 'backtest_paths', 'final')

#### Insights

> *Fill in after running.* Does error grow with the horizon, and where does the model lose to the baseline? Are intermittent series the main weakness? Do forecasts capture the weekly shape and the event peaks, or are they too flat?

---

# Model Interpretation <a name="28"></a>

---

Gain importance (averaged over folds) shows which features the trees rely on, and exact TreeSHAP contributions on the last fold's validation rows show how much each feature moves the forecasts. Importance aggregated by feature family checks the EDA story: if lags and rolling means dominate, history drives the forecast; if events or covariates matter, the business drivers do.

In [ ]:
GBM_RANKED = [m for m in LEADERBOARD.index if m in MODELS]
if not GBM_RANKED:
    note('No boosting model was trained, so there is nothing to interpret.')
else:
    kind = GBM_RANKED[0]
    info = GBM_INFO[kind]
    vm = val_mask(info['last_cutoff'])
    feat_last = FEAT if STRATEGY == 'direct' else build_features(data.assign(y=data['y'].where(data[D] <= info['last_cutoff'])))
    Xs = feat_last.loc[vm, FEATURES]
    Xs = Xs.sample(min(3000, len(Xs)), random_state=CFG.SEED)
    shap_abs = pd.Series(np.abs(info['last_model'].contributions(Xs)).mean(axis=0), index=FEATURES)
    imp = pd.DataFrame({'gain_share': info['importance'], 'mean_abs_shap': shap_abs}).sort_values('mean_abs_shap', ascending=False)
    imp['family'] = [family(c) for c in imp.index]
    imp.to_csv(CFG.FINAL_DIR / 'feature_importance.csv')
    fam = imp.groupby('family')[['gain_share', 'mean_abs_shap']].sum()
    fam = fam / fam.sum()

    fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), gridspec_kw={'width_ratios': [1.5, 1]})
    t = imp.head(20).iloc[::-1]
    fam_colors = {f: PALETTE[i % 10] for i, f in enumerate(sorted(fam.index))}
    axes[0].barh(t.index, t['mean_abs_shap'], color=[fam_colors[f] for f in t['family']])
    axes[0].set_title(f'Top 20 features by mean |SHAP| ({kind})', loc='left')
    handles = [mpl.patches.Patch(color=v, label=k) for k, v in fam_colors.items()]
    axes[0].legend(handles=handles, fontsize=8, loc='lower right')
    f2 = fam.sort_values('mean_abs_shap')
    axes[1].barh(f2.index, f2['mean_abs_shap'], color=[fam_colors[f] for f in f2.index])
    bar_labels(axes[1], '{:.0%}')
    axes[1].set_title('Share of impact by feature family', loc='left')
    suptitle(fig, 'What drives the forecasts?', f'TreeSHAP on {len(Xs):,} rows of the last validation window')
    save_fig(fig, 'feature_importance', 'final')

#### Insights

> *Fill in after running.* Which features and families drive the forecasts? Do event and promo features carry weight, confirming the EDA drivers? Unimportant families can be dropped to simplify the model.

---

# Final Forecast <a name="29"></a>

---

Each component of the selected model is refitted on the full history. Boosting models use the mean best iteration from the backtest, increased by 10% because the final training set is longer. Prediction intervals are empirical: the backtest errors of the selected model, scaled by each series' MASE scale and pooled per forecast step, give the `INTERVAL` quantiles, which are then rescaled to each series. Neighbouring steps are pooled until each window holds at least 40 errors, so a single short series still gets stable quantiles. This makes the intervals honest (they reflect real out-of-sample errors) and widen naturally with the horizon.

In [ ]:
fut_mask = data['is_future']
FC = {}
components = list(WEIGHTS) if BEST == 'ensemble' else [BEST]
for name in components:
    if name in MODELS:
        n_iter = int(np.mean(GBM_INFO[name]['iters']) * 1.1) + 1
        tr = train_mask(LAST)
        m = GBM(name, n_iter=n_iter).fit(FEAT.loc[tr, FEATURES], data.loc[tr, 'y'], volume_weights(tr, LAST))
        if STRATEGY == 'direct':
            pred = pd.Series(m.predict(FEAT.loc[fut_mask, FEATURES]), index=data.index[fut_mask])
        else:
            pred = recursive_predict(m, LAST, list(FUTURE_DATES))
        FC[name] = pd.Series(inv(pred.reindex(data.index[fut_mask]).values), index=data.index[fut_mask])
        print(f'  [{name}] refitted with {n_iter} trees')
    else:
        vals = []
        for s, g in data[fut_mask].groupby('series'):
            fc = baseline_forecast(name, WIDE[s].dropna().values, len(g))
            vals.append(pd.Series(np.clip(fc, 0, None) if NON_NEG else fc, index=g.index))
        FC[name] = pd.concat(vals)
        print(f'  [{name}] forecast from full history')
final_pred = sum(WEIGHTS[n] * FC[n] for n in components) if BEST == 'ensemble' else FC[BEST]

res = BT[BEST].assign(z=(BT[BEST]['y_true'] - BT[BEST]['y_pred']) / BT[BEST]['scale'])
lo_q, hi_q = (1 - CFG.INTERVAL) / 2, 1 - (1 - CFG.INTERVAL) / 2
# pool neighbouring steps until each window holds enough errors for a stable quantile
MIN_ERRORS = 40
steps = np.sort(res['h'].unique())
rows = []
for h in steps:
    w = 0
    while (np.abs(res['h'] - h) <= w).sum() < MIN_ERRORS and w < len(steps):
        w += 1
    rows.append(res.loc[np.abs(res['h'] - h) <= w, 'z'].quantile([lo_q, hi_q]).values)
qz = pd.DataFrame(rows, index=steps, columns=['z_lo', 'z_hi'])
sc_full = scales(LAST)
forecast = data.loc[fut_mask, ['series', D] + IDS + EXOG_KNOWN].copy()
forecast['h'] = forecast.groupby('series').cumcount() + 1
forecast['forecast'] = final_pred.reindex(forecast.index).values
z = qz.reindex(forecast['h']).ffill().values
s_scale = forecast['series'].map(sc_full).values
forecast['lower'] = forecast['forecast'] + z[:, 0] * s_scale
forecast['upper'] = forecast['forecast'] + z[:, 1] * s_scale
if NON_NEG:
    forecast[['lower', 'upper']] = forecast[['lower', 'upper']].clip(lower=0)
for n in components:
    forecast[f'fc_{n}'] = FC[n].reindex(forecast.index).values
covered = BT[BEST].assign(lo=BT[BEST]['y_pred'] + BT[BEST]['h'].map(qz['z_lo']) * BT[BEST]['scale'],
                          hi=BT[BEST]['y_pred'] + BT[BEST]['h'].map(qz['z_hi']) * BT[BEST]['scale'])
COVERAGE = float(((covered['y_true'] >= covered['lo']) & (covered['y_true'] <= covered['hi'])).mean())
print(f'Forecast rows: {len(forecast):,} | {CFG.INTERVAL:.0%} interval in-sample coverage on backtest: {COVERAGE:.1%}')
display(forecast.head())

## Forecast Plots

The aggregate forecast is shown with the recent history, followed by the largest individual series, each with its prediction interval.

In [ ]:
agg_hist = WIDE.sum(axis=1).tail(6 * HORIZON)
agg_fc = forecast.groupby(D)[['forecast', 'lower', 'upper']].sum()
fig, ax = plt.subplots(figsize=(16, 4.8))
ax.plot(agg_hist.index, agg_hist.values, color=PRIMARY, lw=1.4, label='history')
ax.plot(agg_fc.index, agg_fc['forecast'], color=ACCENT, lw=2.2, label=f'forecast ({BEST})')
ax.fill_between(agg_fc.index, agg_fc['lower'], agg_fc['upper'], color=ACCENT, alpha=0.18, label=f'{CFG.INTERVAL:.0%} interval (sum of bounds)')
ax.axvline(LAST, color=MUTED, ls=':', lw=1)
ax.legend(loc='upper left')
date_axis(ax)
suptitle(fig, f'Aggregate forecast: next {HORIZON} steps', f'{len(SERIES)} series summed')
save_fig(fig, 'forecast_aggregate', 'final')

fig, axes = plt.subplots(2, 3, figsize=(18, 7.5), squeeze=False)
for ax, s in zip(axes.ravel(), top):
    h = WIDE[s].tail(4 * HORIZON)
    f = forecast[forecast['series'] == s]
    ax.plot(h.index, h.values, color=PRIMARY, lw=1)
    ax.plot(f[D], f['forecast'], color=ACCENT, lw=2)
    ax.fill_between(f[D], f['lower'], f['upper'], color=ACCENT, alpha=0.18)
    ax.axvline(LAST, color=MUTED, ls=':', lw=1)
    ax.set_title(short(s, 30), loc='left', fontsize=10)
    date_axis(ax)
suptitle(fig, 'Series forecasts', f'top 6 series by volume, {CFG.INTERVAL:.0%} empirical intervals')
save_fig(fig, 'forecast_series', 'final')

#### Insights

> *Fill in after running.* Does the forecast continue the recent level and weekly shape, and does it anticipate known events and promotions in the future frame? Is the interval coverage close to its nominal level?

---

# Export <a name="30"></a>

---

The submission follows the future frame when one is provided (same rows and keys), otherwise the generated future grid. The best baseline's forecast is also written to the baseline folder as a fallback submission. The resolved configuration, decision table, leaderboard, backtests, and a markdown report complete the export.

In [ ]:
def to_submission(frame, col):
    out = frame[IDS + [D, col]].rename(columns={col: Y}) if IDS else frame[[D, col]].rename(columns={col: Y})
    if raw_test is not None:
        keys = IDS + [D]
        base = raw_test.drop(columns=['series'], errors='ignore')
        out = base.merge(out.assign(**{k: out[k].astype(base[k].dtype) for k in IDS}), on=keys, how='left')
        if SAMPLE is not None and SAMPLE.shape[1] >= 2 and SAMPLE.columns[0] in out.columns:
            # follow sample_submission: its id order and its target column name
            sid, tcol = SAMPLE.columns[0], SAMPLE.columns[1]
            out = SAMPLE[[sid]].merge(out[[sid, Y]].rename(columns={Y: tcol}), on=sid, how='left')
    return out

to_submission(forecast, 'forecast').to_csv(CFG.FINAL_DIR / 'submission.csv', index=False)
forecast.to_csv(CFG.FINAL_DIR / 'forecast_full.csv', index=False)

base_fc = data.loc[fut_mask, ['series', D] + IDS].copy()
vals = []
for s, g in base_fc.groupby('series'):
    fc = baseline_forecast(BEST_BASELINE, WIDE[s].dropna().values, len(g))
    vals.append(pd.Series(np.clip(fc, 0, None) if NON_NEG else fc, index=g.index))
base_fc['forecast'] = pd.concat(vals)
to_submission(base_fc, 'forecast').to_csv(CFG.BASELINE_DIR / 'submission.csv', index=False)
to_json({'best_baseline': BEST_BASELINE, 'metric': METRIC, 'leaderboard': base_lb.round(5).to_dict(orient='index')},
        CFG.BASELINE_DIR / 'metrics.json')

settings = {k: v for k, v in vars(Settings).items() if not k.startswith('__') and not callable(v)}
resolved = {'freq': FREQ, 'horizon': HORIZON, 'seasonal_periods': SEASONAL_PERIODS, 'transform': TRANSFORM,
            'fill': FILL, 'strategy': STRATEGY, 'lags': LAGS, 'roll_windows': ROLL_WINDOWS, 'objective': OBJECTIVE,
            'weight_by_volume': WEIGHT_BY_VOLUME, 'metric': METRIC, 'n_folds': N_FOLDS, 'cutoffs': [str(c.date()) for c in CUTOFFS],
            'models': MODELS, 'skipped': SKIPPED, 'best': BEST, 'weights': WEIGHTS, 'interval': CFG.INTERVAL,
            'interval_coverage_backtest': COVERAGE, 'features': FEATURES, 'libraries': AVAILABLE}
to_json({'settings': settings, 'resolved': resolved, 'eda_decisions': EDA}, CFG.FINAL_DIR / 'config.json')

cols = [METRIC, f'{METRIC}_std', 'mase', 'rmse', 'bias']
fmt = lambda v: f'{v:.4f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else '-'
lines = [f'# Forecasting Report: {DATA_PATH.name}', '', '## Setup',
         f'- {len(SERIES)} series of `{Y}` at frequency **{FREQ}**, history to {LAST.date()}, horizon **{HORIZON}**',
         f'- Strategy **{STRATEGY}**, lags {LAGS}, rolling windows {ROLL_WINDOWS}, transform {TRANSFORM}, objective {OBJECTIVE}',
         f'- Backtest: {N_FOLDS} expanding-window folds (cutoffs {", ".join(str(c.date()) for c in CUTOFFS)}), metric **{METRIC}**', '',
         '## Decisions', '', '| setting | value | source | why |', '|---|---|---|---|']
lines += [f'| {s} | `{r.value}` | {r.source} | {r.why} |' for s, r in decision_table.iterrows()]
lines += ['', '## Leaderboard', '', '| model | ' + ' | '.join(cols) + ' |', '|---' * (len(cols) + 1) + '|']
for mname, r in LEADERBOARD.iterrows():
    lines.append(f'| {mname}{" (selected)" if mname == BEST else ""} | ' + ' | '.join(fmt(r.get(c, np.nan)) for c in cols) + ' |')
lines += ['', '## Result',
          f'- Selected **{BEST}**: backtest {METRIC} {LEADERBOARD.loc[BEST, METRIC]:.4f}, {gain:+.1f}% vs best baseline {BEST_BASELINE} ({LEADERBOARD.loc[BEST_BASELINE, METRIC]:.4f})',
          f'- {CFG.INTERVAL:.0%} empirical interval coverage on the backtest: {COVERAGE:.1%}']
if WEIGHTS:
    lines.append('- Ensemble weights: ' + ', '.join(f'{k} {v:.2f}' for k, v in WEIGHTS.items()))
if GBM_RANKED:
    lines += ['', f'## Top features ({GBM_RANKED[0]}, mean |SHAP|)'] + [f'{i + 1}. `{f}` ({r.family})' for i, (f, r) in enumerate(imp.head(10).iterrows())]
report_md = '\n'.join(lines)
(CFG.FINAL_DIR / 'report.md').write_text(report_md, encoding='utf-8')
display(Markdown(report_md))

The last cell prints the output tree for both stages.

In [ ]:
for stage in (CFG.BASELINE_DIR, CFG.FINAL_DIR):
    print(f'{stage}/')
    for p in sorted(stage.rglob('*')):
        if p.is_file():
            print(f'   {str(p.relative_to(stage)):<45} {p.stat().st_size / 1e3:>10,.1f} KB')

#### Insights

> *Fill in after running.* Summarise the selected model, its backtest error and gain over the best baseline, the drivers it relies on, the interval coverage, and how each design choice traces back to the EDA.